## V1

In [ ]:
import os
import cv2
import csv
import numpy as np
import matplotlib.pyplot as plt
from math import degrees
from skimage.measure import approximate_polygon
from sklearn.linear_model import LinearRegression
from skimage.morphology import remove_small_holes, remove_small_objects

# ========= 可調參數 =========
rdp_tolerance = 15.0  # RDP 角點簡化容差（越小角點越多）
direction_thresh_deg = 20.0  # 角度分類門檻（> +20° 視為 Up；< -20° 視為 Down）
extend_scale = 100.0  # 延長擬合線段的像素距離
kernel_size = 7  # 形態學核大小
close_iter = 2  # close 次數
open_iter = 2  # open 次數

# ========= 路徑設定（請修改） =========
input_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB"  # ← 你的輸入資料夾
output_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0810ResultFigures"  # ← 圖片與 CSV 的輸出資料夾
os.makedirs(output_folder, exist_ok=True)
csv_path = os.path.join(output_folder, "batch_edge_summary.csv")


# ========= 工具函式 =========
# def preprocess(gray):
#     """Otsu 二值化 + 形態學清理"""
#     _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
#     binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=close_iter)
#     binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=open_iter)
#     return binary


def preprocess(gray):
    """
    Otsu + 形態學清理 + 去小洞/小物件 + 洪水填洞（flood fill）
    目標：避免前景內部的破洞、邊界斷裂
    """
    # --- 對比增強 & 降噪（可關閉） ---
    # 自覺噪重時，把 clipLimit 調高；不想用可直接註解
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    eq = clahe.apply(gray)
    eq = cv2.GaussianBlur(eq, (5, 5), 0)

    # --- Otsu（用你指定的 BINARY_INV） ---
    _, binary = cv2.threshold(eq, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # --- 初步形態學：先用小核補縫、去毛邊 ---
    k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_small, iterations=2)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k_small, iterations=1)

    # --- 去小洞 / 去小物件（面積門檻依尺寸調整） ---
    # 以白色為前景（BINARY_INV），小洞是黑色區塊；remove_small_holes 需要 bool
    h, w = binary.shape
    area_scale = (h * w) / (1024 * 1024)  # 依影像大小自適應（1024^2 為基準）
    min_hole_area = int(50 * area_scale)  # 內部黑洞小於此面積會被填補
    min_obj_area = int(800 * area_scale)  # 孤立小白點會被去除（保險）

    fg_bool = binary.astype(bool)
    fg_bool = remove_small_holes(fg_bool, area_threshold=min_hole_area)
    fg_bool = remove_small_objects(fg_bool, min_size=min_obj_area)
    binary = (fg_bool.astype(np.uint8)) * 255

    # --- 洪水填洞（最穩健的 hole filling 作法） ---
    im_flood = binary.copy()
    mask = np.zeros((h + 2, w + 2), np.uint8)
    # 從邊界(0,0)填背景到白色
    cv2.floodFill(im_flood, mask, (0, 0), 255)
    # 邊界填白後取反即為「原來的洞」
    holes = cv2.bitwise_not(im_flood)
    # 原圖 OR 洞 → 把洞補回前景
    binary = cv2.bitwise_or(binary, holes)

    # --- 結尾：再做一次你原本的大型 close / open 以達到你喜歡的外觀 ---
    k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_big, iterations=close_iter)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k_big, iterations=open_iter)

    return binary


def find_main_contour(binary):
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    return max(cnts, key=cv2.contourArea)


def nearest_idx(closed_contour, pt):
    d = np.linalg.norm(closed_contour - pt, axis=1)
    return int(np.argmin(d))


def slice_closed_contour(contour, i0, i1):
    """沿輪廓順時針切出 i0→i1 的段（封閉輪廓安全處理）"""
    n = len(contour)
    if n == 0:
        return contour
    if i0 <= i1:
        return contour[i0 : i1 + 1]
    else:
        return np.vstack([contour[i0:], contour[: i1 + 1]])


def fit_segment(points_xy):
    """對一段輪廓點做線性回歸擬合，回傳兩端點（投影）與角度、長度"""
    if len(points_xy) < 2:
        return None
    X = points_xy[:, 0].reshape(-1, 1)
    y = points_xy[:, 1]
    reg = LinearRegression().fit(X, y)
    x1, x2 = float(X[0, 0]), float(X[-1, 0])
    y1 = float(reg.predict([[x1]])[0])
    y2 = float(reg.predict([[x2]])[0])
    angle = degrees(np.arctan2(-(y2 - y1), (x2 - x1)))  # 右=0°, 上=正
    length = np.hypot(x2 - x1, y2 - y1)
    return (x1, y1), (x2, y2), angle, length


def extend_line(p1, p2, scale):
    p1 = np.array(p1, dtype=float)
    p2 = np.array(p2, dtype=float)
    v = p2 - p1
    n = np.linalg.norm(v)
    if n == 0:
        return p1, p2
    v = v / n
    return p1 - v * scale, p2 + v * scale


def classify_dir(angle_deg, th=20.0):
    if angle_deg > th:
        return "Up"
    if angle_deg < -th:
        return "Down"
    return "Flat"


def process_one_image(img_path, save_dir):
    name = os.path.splitext(os.path.basename(img_path))[0]
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return {"file": name, "status": "read_fail"}

    binary = preprocess(gray)
    cnt = find_main_contour(binary)
    if cnt is None:
        return {"file": name, "status": "no_contour"}

    # 原始輪廓點（封閉）
    contour = cnt.reshape(-1, 2).astype(np.float32)

    # RDP 簡化角點（封閉）
    corners = approximate_polygon(contour, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])

    # 逐段擬合
    fitted_segments = []
    for i in range(len(corners) - 1):
        p0, p1 = corners[i], corners[i + 1]
        i0 = nearest_idx(contour, p0)
        i1 = nearest_idx(contour, p1)
        seg_pts = slice_closed_contour(contour, i0, i1)
        res = fit_segment(seg_pts)
        if res is None:
            continue
        q1, q2, ang, L = res
        direction = classify_dir(ang, direction_thresh_deg)
        fitted_segments.append(
            {"p1": q1, "p2": q2, "angle": ang, "length": L, "direction": direction}
        )

    # 主上/下邊
    upper = max(
        (s for s in fitted_segments if s["direction"] == "Up"),
        key=lambda s: s["length"],
        default=None,
    )
    lower = max(
        (s for s in fitted_segments if s["direction"] == "Down"),
        key=lambda s: s["length"],
        default=None,
    )

    # 視覺化
    fig, ax = plt.subplots(figsize=(6, 10))
    ax.imshow(binary, cmap="gray")
    ax.plot(contour[:, 0], contour[:, 1], "g-", alpha=0.25, label="Raw Contour")
    ax.plot(corners[:, 0], corners[:, 1], "ro-", markersize=3, label="Corner Points")

    for seg in fitted_segments:
        color = (
            "blue"
            if seg["direction"] == "Up"
            else ("orange" if seg["direction"] == "Down" else "gray")
        )
        e1, e2 = extend_line(seg["p1"], seg["p2"], extend_scale)
        ax.plot(
            [e1[0], e2[0]], [e1[1], e2[1]], linestyle="--", linewidth=1.6, color=color
        )

    if upper:
        ax.plot(
            [upper["p1"][0], upper["p2"][0]],
            [upper["p1"][1], upper["p2"][1]],
            "r-",
            linewidth=2.5,
        )
        mid = (np.array(upper["p1"]) + np.array(upper["p2"])) / 2
        ax.text(mid[0], mid[1], f"Upper {upper['angle']:.1f}°", color="red", fontsize=9)

    if lower:
        ax.plot(
            [lower["p1"][0], lower["p2"][0]],
            [lower["p1"][1], lower["p2"][1]],
            "lime",
            linewidth=2.5,
        )
        mid = (np.array(lower["p1"]) + np.array(lower["p2"])) / 2
        ax.text(
            mid[0], mid[1], f"Lower {lower['angle']:.1f}°", color="green", fontsize=9
        )

    ax.set_title(name)
    ax.invert_yaxis()
    ax.axis("equal")
    ax.legend(loc="upper left")
    plt.tight_layout()

    out_png = os.path.join(save_dir, f"{name}_fitted_extended.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    return {
        "file": name,
        "status": "ok",
        "n_segments": len(fitted_segments),
        "upper_angle_deg": None if not upper else round(float(upper["angle"]), 3),
        "lower_angle_deg": None if not lower else round(float(lower["angle"]), 3),
        "upper_length_px": None if not upper else round(float(upper["length"]), 3),
        "lower_length_px": None if not lower else round(float(lower["length"]), 3),
        "result_image": out_png,
    }


# ========= 批次主程式 =========
def batch_process(input_dir, save_dir, csv_out):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = [f for f in os.listdir(input_dir) if os.path.splitext(f.lower())[1] in exts]
    files.sort()
    if not files:
        print("⚠️ 找不到圖片檔")
        return

    rows = []
    for i, fname in enumerate(files, 1):
        path = os.path.join(input_dir, fname)
        print(f"[{i}/{len(files)}] {fname} …")
        try:
            res = process_one_image(path, save_dir)
        except Exception as e:
            res = {"file": os.path.splitext(fname)[0], "status": f"error: {e}"}
        rows.append(res)

    # 寫 CSV
    fieldnames = [
        "file",
        "status",
        "n_segments",
        "upper_angle_deg",
        "lower_angle_deg",
        "upper_length_px",
        "lower_length_px",
        "result_image",
    ]
    with open(csv_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)

    print(f"✅ 完成！共處理 {len(files)} 張圖。")
    print(f"📄 CSV：{csv_out}")
    print(f"🖼 圖片輸出資料夾：{save_dir}")


# ========= 執行 =========
if __name__ == "__main__":
    batch_process(input_folder, output_folder, csv_path)

[1/36] 0.3_0.3_30_01_RB(1024).png …
[2/36] 0.3_0.3_30_02_RB(1024).png …
[3/36] 0.3_0.3_60_01_RB(1024).png …
[4/36] 0.3_0.3_60_02_RB(1024).png …
[5/36] 0.3_0.3_90_01_RB(1024).png …
[6/36] 0.3_0.3_90_02_RB(1024).png …
[7/36] 0.3_0.6_30_01_RB(1024).png …
[8/36] 0.3_0.6_60_01_RB(1024).png …
[9/36] 0.3_0.6_60_02_RB(1024).png …
[10/36] 0.3_0.6_90_01_RB(1024).png …
[11/36] 0.3_0.9_30_01-1_RB(1024).png …
[12/36] 0.3_0.9_30_01_RB(1024).png …
[13/36] 0.3_0.9_30_02_RB(1024).png …
[14/36] 0.3_0.9_60_01_RB(1024).png …
[15/36] 0.3_0.9_90_01_RB(1024).png …
[16/36] 0.6_0.3_30_01_RB(1024).png …
[17/36] 0.6_0.3_60_01_RB(1024).png …
[18/36] 0.6_0.3_60_02_RB(1024).png …
[19/36] 0.6_0.3_90_01_RB(1024).png …
[20/36] 0.6_0.3_90_02_RB(1024).png …
[21/36] 0.6_0.6_30_01_RB(1024).png …
[22/36] 0.6_0.6_60_01_RB(1024).png …
[23/36] 0.6_0.6_90_01_RB(1024).png …
[24/36] 0.6_0.9_30_01_RB(1024).png …
[25/36] 0.6_0.9_30_02_RB(1024).png …
[26/36] 0.6_0.9_60_01_RB(1024).png …
[27/36] 0.6_0.9_90_01_RB(1024).png …
[28/36] 

In [25]:
# -*- coding: utf-8 -*-
import os
import re
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt
from math import degrees
from skimage.measure import approximate_polygon
from sklearn.linear_model import LinearRegression
from skimage.morphology import remove_small_holes, remove_small_objects

# ========== 全域參數 ==========
# 幾何擬合/顯示
rdp_tolerance = 15.0
direction_thresh_deg = 20.0
extend_scale = 100.0

# 形態學
kernel_size = 7
close_iter = 2
open_iter = 2

# 尺寸感知補洞（µm²）
hole_area_min_um2 = 2_000.0  # 補掉比這更小的黑洞（雜孔）
hole_area_max_um2 = 100_000.0  # 不補超過這麼大的黑洞（保留腔體）
hole_circularity_min = 0.30  # 只補較圓的洞（避免細長通道被補）

# 失敗時的比例回退（若無法估 px_size），以影像面積比例控制
fallback_max_hole_area_ratio = 0.02  # <=2% 的洞可補
fallback_min_obj_area_ratio = 0.0005  # 去除超小雜點（0.05%）

# I/O
input_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB"
output_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0810ResultFigures"
os.makedirs(output_folder, exist_ok=True)
csv_path = os.path.join(output_folder, "batch_edge_summary.csv")

# Debug 輸出（二值化前/後預檢圖）
debug_binary = True
debug_dir = os.path.join(output_folder, "_debug_binary")
if debug_binary:
    os.makedirs(debug_dir, exist_ok=True)


# ========== 工具：檔名解析 ==========
def parse_name_sizes(filename):
    """
    解析檔名前三段：V_mm(垂直邊)、U_mm(上斜邊)、theta_deg(夾角)
    e.g., '0.9_0.9_90_01_RB(1024).png'  -> (0.9, 0.9, 90.0)
    """
    base = os.path.splitext(os.path.basename(filename))[0]
    toks = base.split("_")
    try:
        V_mm = float(toks[0])
        U_mm = float(toks[1])
        theta = float(toks[2])
        return V_mm, U_mm, theta
    except Exception:
        return None, None, None


# ========== 幾何擬合工具 ==========
def find_main_contour(binary):
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    return max(cnts, key=cv2.contourArea)


def nearest_idx(contour, pt):
    d = np.linalg.norm(contour - pt, axis=1)
    return int(np.argmin(d))


def slice_closed_contour(contour, i0, i1):
    n = len(contour)
    if n == 0:
        return contour
    if i0 <= i1:
        return contour[i0 : i1 + 1]
    else:
        return np.vstack([contour[i0:], contour[: i1 + 1]])


def fit_segment(points_xy):
    if len(points_xy) < 2:
        return None
    X = points_xy[:, 0].reshape(-1, 1)
    y = points_xy[:, 1]
    reg = LinearRegression().fit(X, y)
    x1, x2 = float(X[0, 0]), float(X[-1, 0])
    y1 = float(reg.predict([[x1]])[0])
    y2 = float(reg.predict([[x2]])[0])
    ang = degrees(np.arctan2(-(y2 - y1), (x2 - x1)))  # 右=0°, 上=正
    L = float(np.hypot(x2 - x1, y2 - y1))
    return (x1, y1), (x2, y2), ang, L


def rdp_fit_segments(contour_xy, rdp_tolerance=15.0):
    corners = approximate_polygon(contour_xy, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])
    segs = []
    for i in range(len(corners) - 1):
        i0 = nearest_idx(contour_xy, corners[i])
        i1 = nearest_idx(contour_xy, corners[i + 1])
        pts = slice_closed_contour(contour_xy, i0, i1)
        res = fit_segment(pts)
        if res is not None:
            p1, p2, ang, L = res
            segs.append({"p1": p1, "p2": p2, "angle": ang, "length": L})
    return segs


# ========== 二值化（輕量） ==========
def prelim_binary(gray):
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k, iterations=2)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, k, iterations=1)
    return b


# ========== 由檔名估 µm/px ==========
def estimate_px_size_um(gray, filename, angle_tol_deg=15.0, rdp_tolerance=15.0):
    V_mm, U_mm, theta_deg = parse_name_sizes(filename)
    if V_mm is None:
        return None

    binary0 = prelim_binary(gray)
    cnt = find_main_contour(binary0)
    if cnt is None:
        return None

    contour = cnt.reshape(-1, 2).astype(np.float32)
    segs = rdp_fit_segments(contour, rdp_tolerance=rdp_tolerance)
    if not segs:
        return None

    # 優先用垂直邊（~±90°）
    verts = [s for s in segs if abs(abs(s["angle"]) - 90) <= angle_tol_deg]
    if verts:
        best = max(verts, key=lambda s: s["length"])
        return (V_mm * 1000.0) / best["length"]  # µm/px

    # 否則用上斜邊（角度接近 theta）
    def ang_diff(a, b):
        d = abs(a - b) % 360
        return min(d, 360 - d)

    slopes = [
        s for s in segs if ang_diff(s["angle"] % 180, theta_deg % 180) <= angle_tol_deg
    ]
    if slopes:
        best = max(slopes, key=lambda s: s["length"])
        return (U_mm * 1000.0) / best["length"]

    return None


# ========== 尺寸感知補洞（µm²） ==========
def size_aware_fill(
    binary,
    px_size_um,
    hole_area_min_um2=2_000.0,
    hole_area_max_um2=100_000.0,
    hole_circularity_min=0.30,
    kernel_size=7,
    close_iter=2,
    open_iter=2,
):
    h, w = binary.shape

    # 找洞：背景洪水 → 反相即洞（白）
    im_ff = binary.copy()
    mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(im_ff, mask, (0, 0), 255)
    holes = cv2.bitwise_not(im_ff)

    # µm² → pixel²
    px_per_um2 = 1.0 / (px_size_um**2)
    a_min = int(max(1, round(hole_area_min_um2 * px_per_um2)))
    a_max = int(max(1, round(hole_area_max_um2 * px_per_um2)))

    num, labels, stats, _ = cv2.connectedComponentsWithStats(holes, connectivity=8)
    holes_keep = np.zeros_like(binary)
    for i in range(1, num):
        area_px = int(stats[i, cv2.CC_STAT_AREA])
        if not (a_min <= area_px <= a_max):
            continue
        mask_i = (labels == i).astype(np.uint8)
        cnts, _ = cv2.findContours(mask_i, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not cnts:
            continue
        peri = cv2.arcLength(cnts[0], True)
        if peri <= 0:
            continue
        circularity = 4.0 * np.pi * area_px / (peri * peri)
        if circularity < hole_circularity_min:
            continue
        holes_keep[labels == i] = 255

    filled = cv2.bitwise_or(binary, holes_keep)

    # 收尾：大核 close/open
    k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    filled = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, k_big, iterations=close_iter)
    filled = cv2.morphologyEx(filled, cv2.MORPH_OPEN, k_big, iterations=open_iter)
    return filled


# ========== 最終預處理：二段式 + 尺度回退 ==========
def preprocess_final(gray, filename):
    # Pass A：輕量二值化（不補洞）
    b0 = prelim_binary(gray)

    # 估 µm/px
    px_um = estimate_px_size_um(gray, filename)

    if px_um is None:
        # 回退：用相對比例處理（不依賴實際尺寸）
        h, w = b0.shape
        area = h * w
        fg = b0.astype(bool)
        fg = remove_small_objects(
            fg, min_size=max(1, int(fallback_min_obj_area_ratio * area))
        )
        fg = remove_small_holes(
            fg, area_threshold=int(0.25 * fallback_max_hole_area_ratio * area)
        )
        b0_small = (fg.astype(np.uint8)) * 255

        # 只補「小洞」（<= ratio）
        im_ff = b0_small.copy()
        mask = np.zeros((h + 2, w + 2), np.uint8)
        cv2.floodFill(im_ff, mask, (0, 0), 255)
        holes = cv2.bitwise_not(im_ff)

        num, labels = cv2.connectedComponents(holes, connectivity=8)
        a_max = int(fallback_max_hole_area_ratio * area)
        holes_keep = np.zeros_like(b0_small)
        for i in range(1, num):
            if (labels == i).sum() <= a_max:
                holes_keep[labels == i] = 255
        filled = cv2.bitwise_or(b0_small, holes_keep)
    else:
        # Pass B：尺寸感知補洞
        filled = size_aware_fill(
            b0,
            px_size_um=px_um,
            hole_area_min_um2=hole_area_min_um2,
            hole_area_max_um2=hole_area_max_um2,
            hole_circularity_min=hole_circularity_min,
            kernel_size=kernel_size,
            close_iter=close_iter,
            open_iter=open_iter,
        )

    return filled, px_um


# ========== 方向分類 ==========
def classify_dir(angle_deg, th=20.0):
    if angle_deg > th:
        return "Up"
    if angle_deg < -th:
        return "Down"
    return "Flat"


def extend_line(p1, p2, scale):
    p1 = np.array(p1, dtype=float)
    p2 = np.array(p2, dtype=float)
    v = p2 - p1
    n = np.linalg.norm(v)
    if n == 0:
        return p1, p2
    v = v / n
    return p1 - v * scale, p2 + v * scale


# ========== 單張流程 ==========
def process_one_image(img_path, save_dir):
    name = os.path.splitext(os.path.basename(img_path))[0]
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return {"file": name, "status": "read_fail"}

    # 預處理（尺寸感知補洞）
    binary, px_um = preprocess_final(gray, name)

    if debug_binary:
        cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)

    cnt = find_main_contour(binary)
    if cnt is None:
        return {"file": name, "status": "no_contour"}

    contour = cnt.reshape(-1, 2).astype(np.float32)

    # RDP → 區段擬合
    corners = approximate_polygon(contour, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])

    fitted_segments = []
    for i in range(len(corners) - 1):
        p0, p1 = corners[i], corners[i + 1]
        i0 = nearest_idx(contour, p0)
        i1 = nearest_idx(contour, p1)
        seg_pts = slice_closed_contour(contour, i0, i1)
        res = fit_segment(seg_pts)
        if res is None:
            continue
        q1, q2, ang, L = res
        fitted_segments.append(
            {
                "p1": q1,
                "p2": q2,
                "angle": ang,
                "length": L,
                "direction": classify_dir(ang, direction_thresh_deg),
            }
        )

    # 主上/下邊
    upper = max(
        (s for s in fitted_segments if s["direction"] == "Up"),
        key=lambda s: s["length"],
        default=None,
    )
    lower = max(
        (s for s in fitted_segments if s["direction"] == "Down"),
        key=lambda s: s["length"],
        default=None,
    )

    # 視覺化
    fig, ax = plt.subplots(figsize=(6, 10))
    ax.imshow(binary, cmap="gray")
    ax.plot(contour[:, 0], contour[:, 1], "g-", alpha=0.25, label="Raw Contour")
    ax.plot(corners[:, 0], corners[:, 1], "ro-", markersize=3, label="Corner Points")

    for seg in fitted_segments:
        color = (
            "blue"
            if seg["direction"] == "Up"
            else ("orange" if seg["direction"] == "Down" else "gray")
        )
        e1, e2 = extend_line(seg["p1"], seg["p2"], extend_scale)
        ax.plot(
            [e1[0], e2[0]], [e1[1], e2[1]], linestyle="--", linewidth=1.6, color=color
        )

    if upper:
        ax.plot(
            [upper["p1"][0], upper["p2"][0]],
            [upper["p1"][1], upper["p2"][1]],
            "r-",
            linewidth=2.5,
        )
        mid = (np.array(upper["p1"]) + np.array(upper["p2"])) / 2
        ax.text(mid[0], mid[1], f"Upper {upper['angle']:.1f}°", color="red", fontsize=9)

    if lower:
        ax.plot(
            [lower["p1"][0], lower["p2"][0]],
            [lower["p1"][1], lower["p2"][1]],
            "lime",
            linewidth=2.5,
        )
        mid = (np.array(lower["p1"]) + np.array(lower["p2"])) / 2
        ax.text(
            mid[0], mid[1], f"Lower {lower['angle']:.1f}°", color="green", fontsize=9
        )

    ax.set_title(name)
    ax.invert_yaxis()
    ax.axis("equal")
    ax.legend(loc="upper left")
    plt.tight_layout()

    out_png = os.path.join(save_dir, f"{name}_fitted_extended.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    return {
        "file": name,
        "status": "ok",
        "px_um": None if px_um is None else round(float(px_um), 6),
        "n_segments": len(fitted_segments),
        "upper_angle_deg": None if not upper else round(float(upper["angle"]), 3),
        "lower_angle_deg": None if not lower else round(float(lower["angle"]), 3),
        "upper_length_px": None if not upper else round(float(upper["length"]), 3),
        "lower_length_px": None if not lower else round(float(lower["length"]), 3),
        "result_image": out_png,
    }


# ========== 批次 ==========
def batch_process(input_dir, save_dir, csv_out):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = [f for f in os.listdir(input_dir) if os.path.splitext(f.lower())[1] in exts]
    files.sort()
    if not files:
        print("⚠️ 找不到圖片檔")
        return

    rows = []
    for i, fname in enumerate(files, 1):
        path = os.path.join(input_dir, fname)
        print(f"[{i}/{len(files)}] {fname} …")
        try:
            res = process_one_image(path, save_dir)
        except Exception as e:
            res = {"file": os.path.splitext(fname)[0], "status": f"error: {e}"}
        rows.append(res)

    fieldnames = [
        "file",
        "status",
        "px_um",
        "n_segments",
        "upper_angle_deg",
        "lower_angle_deg",
        "upper_length_px",
        "lower_length_px",
        "result_image",
    ]
    with open(csv_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)

    print(f"✅ 完成！共處理 {len(files)} 張圖。")
    print(f"📄 CSV：{csv_out}")
    print(f"🖼 圖片輸出資料夾：{save_dir}")
    if debug_binary:
        print(f"🧪 Debug 二值化輸出：{debug_dir}")


# ========== 執行 ==========
if __name__ == "__main__":
    batch_process(input_folder, output_folder, csv_path)

[1/36] 0.3_0.3_30_01_RB(1024).png …
[2/36] 0.3_0.3_30_02_RB(1024).png …
[3/36] 0.3_0.3_60_01_RB(1024).png …
[4/36] 0.3_0.3_60_02_RB(1024).png …
[5/36] 0.3_0.3_90_01_RB(1024).png …
[6/36] 0.3_0.3_90_02_RB(1024).png …
[7/36] 0.3_0.6_30_01_RB(1024).png …
[8/36] 0.3_0.6_60_01_RB(1024).png …
[9/36] 0.3_0.6_60_02_RB(1024).png …
[10/36] 0.3_0.6_90_01_RB(1024).png …
[11/36] 0.3_0.9_30_01-1_RB(1024).png …
[12/36] 0.3_0.9_30_01_RB(1024).png …
[13/36] 0.3_0.9_30_02_RB(1024).png …
[14/36] 0.3_0.9_60_01_RB(1024).png …
[15/36] 0.3_0.9_90_01_RB(1024).png …
[16/36] 0.6_0.3_30_01_RB(1024).png …
[17/36] 0.6_0.3_60_01_RB(1024).png …
[18/36] 0.6_0.3_60_02_RB(1024).png …
[19/36] 0.6_0.3_90_01_RB(1024).png …
[20/36] 0.6_0.3_90_02_RB(1024).png …
[21/36] 0.6_0.6_30_01_RB(1024).png …
[22/36] 0.6_0.6_60_01_RB(1024).png …
[23/36] 0.6_0.6_90_01_RB(1024).png …
[24/36] 0.6_0.9_30_01_RB(1024).png …
[25/36] 0.6_0.9_30_02_RB(1024).png …
[26/36] 0.6_0.9_60_01_RB(1024).png …
[27/36] 0.6_0.9_90_01_RB(1024).png …
[28/36] 

## V2 count_components

In [30]:
# -*- coding: utf-8 -*-
import os
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt
from math import degrees
from skimage.measure import approximate_polygon
from sklearn.linear_model import LinearRegression
from skimage.morphology import remove_small_holes, remove_small_objects
from scipy.ndimage import binary_fill_holes

# ======================= 全域參數 =======================
# 幾何擬合/顯示
rdp_tolerance = 15.0
direction_thresh_deg = 20.0
extend_scale = 100.0

# 形態學（kernel 固定，不自動放大）
kernel_size = 7
close_iter = 2
open_iter = 2

# 尺寸感知補洞（µm²）
hole_area_min_um2 = 2_000.0
hole_area_max_um2 = 100_000.0
hole_circularity_min = 0.30

# 回退比例（無法估 px_size 時）
fallback_max_hole_area_ratio = 0.02
fallback_min_obj_area_ratio = 0.0005

# 連通域自動調參（只增 close、不增 kernel）
max_components = 1
min_component_area_ratio = 0.001
max_retries = 50
close_step = 6

# 主體有效性門檻（避免 components==1 但其實是雜訊）
min_main_area_ratio = 0.02
min_main_short_side = 80

# 是否保留左側細長基板（建議 True）
keep_left_substrate = False

# I/O
input_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB"
output_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0810ResultFigures"
os.makedirs(output_folder, exist_ok=True)
csv_path = os.path.join(output_folder, "batch_edge_summary.csv")

# Debug 二值圖
debug_binary = True
debug_dir = os.path.join(output_folder, "_debug_binary")
if debug_binary:
    os.makedirs(debug_dir, exist_ok=True)


# ======================= 小工具 =======================
def parse_name_sizes(filename):
    base = os.path.splitext(os.path.basename(filename))[0]
    toks = base.split("_")
    try:
        V_mm = float(toks[0])
        U_mm = float(toks[1])
        theta = float(toks[2])
        return V_mm, U_mm, theta
    except Exception:
        return None, None, None


def prelim_binary(gray):
    # 單純 Otsu + 小型態學（就是你之前穩的那版）
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k, iterations=2)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, k, iterations=1)
    return b


def count_components(binary, min_area_ratio=0.001):
    h, w = binary.shape
    area = h * w
    min_area = max(1, int(min_area_ratio * area))
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    return sum(int(stats[i, cv2.CC_STAT_AREA] >= min_area) for i in range(1, num))


def is_valid_main_object(binary):
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num <= 1:
        return False
    idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    h, w = binary.shape
    img_area = h * w
    main_area = stats[idx, cv2.CC_STAT_AREA]
    bw = stats[idx, cv2.CC_STAT_WIDTH]
    bh = stats[idx, cv2.CC_STAT_HEIGHT]
    return (main_area / img_area) >= min_main_area_ratio and min(
        int(bw), int(bh)
    ) >= int(min_main_short_side)


def find_main_contour(binary):
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return max(cnts, key=cv2.contourArea) if cnts else None


def nearest_idx(contour, pt):
    d = np.linalg.norm(contour - pt, axis=1)
    return int(np.argmin(d))


def slice_closed_contour(contour, i0, i1):
    n = len(contour)
    return (
        contour[i0 : i1 + 1]
        if i0 <= i1
        else np.vstack([contour[i0:], contour[: i1 + 1]])
    )


def fit_segment(points_xy):
    if len(points_xy) < 2:
        return None
    X = points_xy[:, 0].reshape(-1, 1)
    y = points_xy[:, 1]
    reg = LinearRegression().fit(X, y)
    x1, x2 = float(X[0, 0]), float(X[-1, 0])
    y1 = float(reg.predict([[x1]])[0])
    y2 = float(reg.predict([[x2]])[0])
    ang = degrees(np.arctan2(-(y2 - y1), (x2 - x1)))
    L = float(np.hypot(x2 - x1, y2 - y1))
    return (x1, y1), (x2, y2), ang, L


def rdp_fit_segments(contour_xy, rdp_tolerance=15.0):
    corners = approximate_polygon(contour_xy, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])
    segs = []
    for i in range(len(corners) - 1):
        i0 = nearest_idx(contour_xy, corners[i])
        i1 = nearest_idx(contour_xy, corners[i + 1])
        pts = slice_closed_contour(contour_xy, i0, i1)
        res = fit_segment(pts)
        if res is not None:
            p1, p2, ang, L = res
            segs.append({"p1": p1, "p2": p2, "angle": ang, "length": L})
    return segs


def estimate_px_size_um(gray, filename, angle_tol_deg=15.0):
    V_mm, U_mm, theta_deg = parse_name_sizes(filename)
    if V_mm is None:
        return None
    binary0 = prelim_binary(gray)
    cnt = find_main_contour(binary0)
    if cnt is None:
        return None
    contour = cnt.reshape(-1, 2).astype(np.float32)
    segs = rdp_fit_segments(contour, rdp_tolerance)

    if not segs:
        return None
    verts = [s for s in segs if abs(abs(s["angle"]) - 90) <= angle_tol_deg]
    if verts:
        best = max(verts, key=lambda s: s["length"])
        return (V_mm * 1000.0) / best["length"]

    def ang_diff(a, b):
        d = abs(a - b) % 360
        return min(d, 360 - d)

    slopes = [
        s for s in segs if ang_diff(s["angle"] % 180, theta_deg % 180) <= angle_tol_deg
    ]
    if slopes:
        best = max(slopes, key=lambda s: s["length"])
        return (U_mm * 1000.0) / best["length"]
    return None


def size_aware_fill(binary, px_size_um):
    h, w = binary.shape
    im_ff = binary.copy()
    mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(im_ff, mask, (0, 0), 255)
    holes = cv2.bitwise_not(im_ff)

    px_per_um2 = 1.0 / (px_size_um**2)
    a_min = int(max(1, round(hole_area_min_um2 * px_per_um2)))
    a_max = int(max(1, round(hole_area_max_um2 * px_per_um2)))

    num, labels, stats, _ = cv2.connectedComponentsWithStats(holes, connectivity=8)
    holes_keep = np.zeros_like(binary)
    for i in range(1, num):
        area_px = int(stats[i, cv2.CC_STAT_AREA])
        if not (a_min <= area_px <= a_max):
            continue
        mask_i = (labels == i).astype(np.uint8)
        cnts, _ = cv2.findContours(mask_i, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not cnts:
            continue
        peri = cv2.arcLength(cnts[0], True)
        if peri <= 0:
            continue
        circ = 4.0 * np.pi * area_px / (peri * peri)
        if circ < hole_circularity_min:
            continue
        holes_keep[labels == i] = 255

    filled = cv2.bitwise_or(binary, holes_keep)
    k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    filled = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, k_big, iterations=close_iter)
    filled = cv2.morphologyEx(filled, cv2.MORPH_OPEN, k_big, iterations=open_iter)
    return filled


def keep_main_and_left(binary):
    if not keep_left_substrate:
        return binary
    h, w = binary.shape
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num <= 1:
        return binary
    # 主體
    areas = stats[1:, cv2.CC_STAT_AREA]
    main_id = 1 + np.argmax(areas)
    keep = {main_id}
    # 左側細長基板
    for i in range(1, num):
        if i == main_id:
            continue
        x, y, ww, hh, area = (
            stats[i, 0],
            stats[i, 1],
            stats[i, 2],
            stats[i, 3],
            stats[i, cv2.CC_STAT_AREA],
        )
        touches_left = x == 0
        tall_thin = (hh >= 0.40 * h) and (ww <= 0.15 * w)
        in_left = (x + ww) <= int(0.30 * w)
        big_enough = area >= 0.005 * (h * w)
        if touches_left and tall_thin and in_left and big_enough:
            keep.add(i)
    mask = np.isin(labels, list(keep))
    mask = binary_fill_holes(mask)
    return (mask.astype(np.uint8)) * 255


def preprocess_final(gray, filename):
    b0 = prelim_binary(gray)
    px_um = estimate_px_size_um(gray, filename)

    def run_once(basis_binary, k, c_iter, o_iter):
        if px_um is None:
            # 回退：僅做保守的去小物件/小洞 + 很小的補洞上限
            h, w = basis_binary.shape
            area = h * w
            fg = basis_binary.astype(bool)
            fg = remove_small_objects(
                fg, min_size=max(1, int(fallback_min_obj_area_ratio * area))
            )
            fg = remove_small_holes(
                fg, area_threshold=int(0.25 * fallback_max_hole_area_ratio * area)
            )
            b_small = (fg.astype(np.uint8)) * 255

            im_ff = b_small.copy()
            mask = np.zeros((h + 2, w + 2), np.uint8)
            cv2.floodFill(im_ff, mask, (0, 0), 255)
            holes = cv2.bitwise_not(im_ff)

            num, labels = cv2.connectedComponents(holes, connectivity=8)
            a_max = int(fallback_max_hole_area_ratio * area)
            holes_keep = np.zeros_like(b_small)
            for i in range(1, num):
                if (labels == i).sum() <= a_max:
                    holes_keep[labels == i] = 255
            filled = cv2.bitwise_or(b_small, holes_keep)
        else:
            filled = size_aware_fill(basis_binary, px_um)
        return filled

    k_used = kernel_size
    close_used = close_iter
    open_used = open_iter
    binary = run_once(b0, k_used, close_used, open_used)
    comp = count_components(binary, min_component_area_ratio)

    tries = 0
    while comp > max_components and tries < max_retries:
        tries += 1
        close_used += close_step
        binary = run_once(b0, k_used, close_used, open_used)
        comp = count_components(binary, min_component_area_ratio)

    if keep_left_substrate:
        binary = keep_main_and_left(binary)

    meta = {
        "px_um": px_um,
        "components": comp,
        "close_iter_used": close_used,
        "kernel_used": k_used,
    }
    return binary, px_um, meta


def classify_dir(angle_deg, th=20.0):
    if angle_deg > th:
        return "Up"
    if angle_deg < -th:
        return "Down"
    return "Flat"


def extend_line(p1, p2, scale):
    p1 = np.array(p1, dtype=float)
    p2 = np.array(p2, dtype=float)
    v = p2 - p1
    n = np.linalg.norm(v)
    if n == 0:
        return p1, p2
    v = v / n
    return p1 - v * scale, p2 + v * scale


# ======================= 單張流程 =======================
def process_one_image(img_path, save_dir):
    name = os.path.splitext(os.path.basename(img_path))[0]
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return {"file": name, "status": "read_fail"}

    binary, px_um, meta = preprocess_final(gray, name)

    if meta.get("components", 0) <= 1 and not is_valid_main_object(binary):
        if debug_binary:
            cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)
        return {
            "file": name,
            "status": "noise_or_too_small",
            "px_um": None if px_um is None else round(float(px_um), 6),
            "n_segments": "",
            "upper_angle_deg": "",
            "lower_angle_deg": "",
            "upper_length_px": "",
            "lower_length_px": "",
            "components": meta.get("components", ""),
            "close_iter_used": meta.get("close_iter_used", ""),
            "kernel_used": meta.get("kernel_used", ""),
            "result_image": "",
        }

    if debug_binary:
        cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)

    cnt = find_main_contour(binary)
    if cnt is None:
        return {"file": name, "status": "no_contour"}

    contour = cnt.reshape(-1, 2).astype(np.float32)
    corners = approximate_polygon(contour, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])

    fitted_segments = []
    for i in range(len(corners) - 1):
        p0, p1 = corners[i], corners[i + 1]
        i0 = nearest_idx(contour, p0)
        i1 = nearest_idx(contour, p1)
        seg_pts = slice_closed_contour(contour, i0, i1)
        res = fit_segment(seg_pts)
        if res is None:
            continue
        q1, q2, ang, L = res
        fitted_segments.append(
            {
                "p1": q1,
                "p2": q2,
                "angle": ang,
                "length": L,
                "direction": classify_dir(ang, direction_thresh_deg),
            }
        )

    upper = max(
        (s for s in fitted_segments if s["direction"] == "Up"),
        key=lambda s: s["length"],
        default=None,
    )
    lower = max(
        (s for s in fitted_segments if s["direction"] == "Down"),
        key=lambda s: s["length"],
        default=None,
    )

    # 視覺化
    fig, ax = plt.subplots(figsize=(6, 10))
    ax.imshow(binary, cmap="gray")
    ax.plot(contour[:, 0], contour[:, 1], "g-", alpha=0.25, label="Raw Contour")
    ax.plot(corners[:, 0], corners[:, 1], "ro-", markersize=3, label="Corner Points")

    for seg in fitted_segments:
        color = (
            "blue"
            if seg["direction"] == "Up"
            else ("orange" if seg["direction"] == "Down" else "gray")
        )
        e1, e2 = extend_line(seg["p1"], seg["p2"], extend_scale)
        ax.plot(
            [e1[0], e2[0]], [e1[1], e2[1]], linestyle="--", linewidth=1.6, color=color
        )

    if upper:
        ax.plot(
            [upper["p1"][0], upper["p2"][0]],
            [upper["p1"][1], upper["p2"][1]],
            "r-",
            linewidth=2.5,
        )
        mid = (np.array(upper["p1"]) + np.array(upper["p2"])) / 2
        ax.text(mid[0], mid[1], f"Upper {upper['angle']:.1f}°", color="red", fontsize=9)

    if lower:
        ax.plot(
            [lower["p1"][0], lower["p2"][0]],
            [lower["p1"][1], lower["p2"][1]],
            "lime",
            linewidth=2.5,
        )
        mid = (np.array(lower["p1"]) + np.array(lower["p2"])) / 2
        ax.text(
            mid[0], mid[1], f"Lower {lower['angle']:.1f}°", color="green", fontsize=9
        )

    ax.text(
        0.02,
        0.98,
        f"comp={meta.get('components','?')}, close={meta.get('close_iter_used','?')}, k={meta.get('kernel_used','?')}",
        color="cyan",
        fontsize=10,
        bbox=dict(facecolor="gray", alpha=0.3),
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    ax.set_title(name)
    # 讓坐標系與影像一致：左上角 (0,0)，y 向下增加
    h, w = binary.shape
    ax.imshow(binary, cmap="gray", origin="upper")
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)          # 這行把 y=0 放在頂端，往下遞增
    ax.set_aspect("equal")     # 等效於 axis("equal")，但不會再翻轉座標
    ax.legend(loc="upper left")
    plt.tight_layout()

    out_png = os.path.join(save_dir, f"{name}_fitted_extended.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    return {
        "file": name,
        "status": "ok",
        "px_um": None if px_um is None else round(float(px_um), 6),
        "n_segments": len(fitted_segments),
        "upper_angle_deg": None if not upper else round(float(upper["angle"]), 3),
        "lower_angle_deg": None if not lower else round(float(lower["angle"]), 3),
        "upper_length_px": None if not upper else round(float(upper["length"]), 3),
        "lower_length_px": None if not lower else round(float(lower["length"]), 3),
        "components": meta.get("components", ""),
        "close_iter_used": meta.get("close_iter_used", ""),
        "kernel_used": meta.get("kernel_used", ""),
        "result_image": out_png,
    }


# ======================= 批次流程 =======================
def batch_process(input_dir, save_dir, csv_out):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = [f for f in os.listdir(input_dir) if os.path.splitext(f.lower())[1] in exts]
    files.sort()
    if not files:
        print("⚠️ 找不到圖片檔")
        return

    rows = []
    for i, fname in enumerate(files, 1):
        path = os.path.join(input_dir, fname)
        print(f"[{i}/{len(files)}] {fname} …")
        try:
            res = process_one_image(path, save_dir)
        except Exception as e:
            res = {"file": os.path.splitext(fname)[0], "status": f"error: {e}"}
        rows.append(res)

    fieldnames = [
        "file",
        "status",
        "px_um",
        "n_segments",
        "upper_angle_deg",
        "lower_angle_deg",
        "upper_length_px",
        "lower_length_px",
        "components",
        "close_iter_used",
        "kernel_used",
        "result_image",
    ]
    with open(csv_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fieldnames})

    print(f"✅ 完成！共處理 {len(files)} 張圖。")
    print(f"📄 CSV：{csv_out}")
    print(f"🖼 圖片輸出資料夾：{save_dir}")
    if debug_binary:
        print(f"🧪 Debug 二值化輸出：{debug_dir}")


# ======================= 執行 =======================
if __name__ == "__main__":
    batch_process(input_folder, output_folder, csv_path)

[1/33] 0.3_0.3_30_01_RB(1024).png …
[2/33] 0.3_0.3_30_02_RB(1024).png …
[3/33] 0.3_0.3_60_01_RB(1024).png …
[4/33] 0.3_0.3_60_02_RB(1024).png …
[5/33] 0.3_0.3_90_01_RB(1024).png …
[6/33] 0.3_0.3_90_02_RB(1024).png …
[7/33] 0.3_0.6_30_01_RB(1024).png …
[8/33] 0.3_0.6_60_01_RB(1024).png …
[9/33] 0.3_0.6_60_02_RB(1024).png …
[10/33] 0.3_0.6_90_01_RB(1024).png …
[11/33] 0.3_0.9_30_02_RB(1024).png …
[12/33] 0.3_0.9_60_01_RB(1024).png …
[13/33] 0.3_0.9_90_01_RB(1024).png …
[14/33] 0.6_0.3_30_01_RB(1024).png …
[15/33] 0.6_0.3_60_01_RB(1024).png …
[16/33] 0.6_0.3_60_02_RB(1024).png …
[17/33] 0.6_0.3_90_01_RB(1024).png …
[18/33] 0.6_0.3_90_02_RB(1024).png …
[19/33] 0.6_0.6_30_01_RB(1024).png …
[20/33] 0.6_0.6_60_01_RB(1024).png …
[21/33] 0.6_0.6_90_01_RB(1024).png …
[22/33] 0.6_0.9_30_02_RB(1024).png …
[23/33] 0.6_0.9_60_01_RB(1024).png …
[24/33] 0.6_0.9_90_01_RB(1024).png …
[25/33] 0.9_0.3_30_01_RB(1024).png …
[26/33] 0.9_0.3_60_06_RB(1024).png …
[27/33] 0.9_0.3_90_01_RB(1024).png …
[28/33] 0.

## V3 REAL LENGTH

In [56]:
# -*- coding: utf-8 -*-
import os
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt
from math import degrees
from skimage.measure import approximate_polygon
from sklearn.linear_model import LinearRegression
from skimage.morphology import remove_small_holes, remove_small_objects
from scipy.ndimage import binary_fill_holes

# ======================= 全域參數 =======================
# 幾何擬合/顯示
rdp_tolerance = 15.0
direction_thresh_deg = 20.0
extend_scale = 100.0

# 形態學（kernel 固定，不自動放大）
kernel_size = 7
close_iter = 2
open_iter = 2

# 尺寸感知補洞（µm²）
hole_area_min_um2 = 2_000.0
hole_area_max_um2 = 100_000.0
hole_circularity_min = 0.30

# 回退比例（無法估 px_size 時）
fallback_max_hole_area_ratio = 0.02
fallback_min_obj_area_ratio = 0.0005

# 連通域自動調參（只增 close、不增 kernel）
max_components = 1
min_component_area_ratio = 0.001
max_retries = 50
close_step = 6

# 主體有效性門檻（避免 components==1 但其實是雜訊）
min_main_area_ratio = 0.02
min_main_short_side = 80

# 是否保留左側細長基板
keep_left_substrate = False

# I/O
input_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB"
output_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0810ResultFigures"
os.makedirs(output_folder, exist_ok=True)
csv_path = os.path.join(output_folder, "batch_edge_summary.csv")

# Debug 二值圖
debug_binary = True
debug_dir = os.path.join(output_folder, "_debug_binary")
if debug_binary:
    os.makedirs(debug_dir, exist_ok=True)


# ======================= 小工具 =======================
def parse_name_sizes(filename):
    base = os.path.splitext(os.path.basename(filename))[0]
    toks = base.split("_")
    try:
        V_mm = float(toks[0])
        U_mm = float(toks[1])
        theta = float(toks[2])
        return V_mm, U_mm, theta
    except Exception:
        return None, None, None


def prelim_binary(gray):
    # 你的穩定版：Otsu + 小型態學
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k, iterations=2)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, k, iterations=1)
    return b


def count_components(binary, min_area_ratio=0.001):
    h, w = binary.shape
    area = h * w
    min_area = max(1, int(min_area_ratio * area))
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    return sum(int(stats[i, cv2.CC_STAT_AREA] >= min_area) for i in range(1, num))


def is_valid_main_object(binary):
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num <= 1:
        return False
    idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    h, w = binary.shape
    img_area = h * w
    main_area = stats[idx, cv2.CC_STAT_AREA]
    bw = stats[idx, cv2.CC_STAT_WIDTH]
    bh = stats[idx, cv2.CC_STAT_HEIGHT]
    return (main_area / img_area) >= min_main_area_ratio and min(
        int(bw), int(bh)
    ) >= int(min_main_short_side)


def find_main_contour(binary):
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return max(cnts, key=cv2.contourArea) if cnts else None


def nearest_idx(contour, pt):
    d = np.linalg.norm(contour - pt, axis=1)
    return int(np.argmin(d))


def slice_closed_contour(contour, i0, i1):
    n = len(contour)
    return (
        contour[i0 : i1 + 1]
        if i0 <= i1
        else np.vstack([contour[i0:], contour[: i1 + 1]])
    )


def fit_segment(points_xy):
    if len(points_xy) < 2:
        return None
    X = points_xy[:, 0].reshape(-1, 1)
    y = points_xy[:, 1]
    reg = LinearRegression().fit(X, y)
    x1, x2 = float(X[0, 0]), float(X[-1, 0])
    y1 = float(reg.predict([[x1]])[0])
    y2 = float(reg.predict([[x2]])[0])
    ang = degrees(np.arctan2(-(y2 - y1), (x2 - x1)))  # 右=0°, 上=正
    L = float(np.hypot(x2 - x1, y2 - y1))
    return (x1, y1), (x2, y2), ang, L


def rdp_fit_segments(contour_xy, rdp_tolerance=15.0):
    corners = approximate_polygon(contour_xy, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])
    segs = []
    for i in range(len(corners) - 1):
        i0 = nearest_idx(contour_xy, corners[i])
        i1 = nearest_idx(contour_xy, corners[i + 1])
        pts = slice_closed_contour(contour_xy, i0, i1)
        res = fit_segment(pts)
        if res is not None:
            p1, p2, ang, L = res
            segs.append({"p1": p1, "p2": p2, "angle": ang, "length": L})
    return segs


def estimate_px_size_um(gray, filename, angle_tol_deg=15.0):
    V_mm, U_mm, theta_deg = parse_name_sizes(filename)
    if V_mm is None:
        return None
    binary0 = prelim_binary(gray)
    cnt = find_main_contour(binary0)
    if cnt is None:
        return None
    contour = cnt.reshape(-1, 2).astype(np.float32)
    segs = rdp_fit_segments(contour, rdp_tolerance)
    if not segs:
        return None

    verts = [s for s in segs if abs(abs(s["angle"]) - 90) <= angle_tol_deg]
    if verts:
        best = max(verts, key=lambda s: s["length"])
        return (V_mm * 1000.0) / best["length"]

    def ang_diff(a, b):
        d = abs(a - b) % 360
        return min(d, 360 - d)

    slopes = [
        s for s in segs if ang_diff(s["angle"] % 180, theta_deg % 180) <= angle_tol_deg
    ]
    if slopes:
        best = max(slopes, key=lambda s: s["length"])
        return (U_mm * 1000.0) / best["length"]
    return None


def size_aware_fill(binary, px_size_um):
    h, w = binary.shape
    im_ff = binary.copy()
    mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(im_ff, mask, (0, 0), 255)
    holes = cv2.bitwise_not(im_ff)

    px_per_um2 = 1.0 / (px_size_um**2)
    a_min = int(max(1, round(hole_area_min_um2 * px_per_um2)))
    a_max = int(max(1, round(hole_area_max_um2 * px_per_um2)))

    num, labels, stats, _ = cv2.connectedComponentsWithStats(holes, connectivity=8)
    holes_keep = np.zeros_like(binary)
    for i in range(1, num):
        area_px = int(stats[i, cv2.CC_STAT_AREA])
        if not (a_min <= area_px <= a_max):
            continue
        mask_i = (labels == i).astype(np.uint8)
        cnts, _ = cv2.findContours(mask_i, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not cnts:
            continue
        peri = cv2.arcLength(cnts[0], True)
        if peri <= 0:
            continue
        circ = 4.0 * np.pi * area_px / (peri * peri)
        if circ < hole_circularity_min:
            continue
        holes_keep[labels == i] = 255

    filled = cv2.bitwise_or(binary, holes_keep)
    k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    filled = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, k_big, iterations=close_iter)
    filled = cv2.morphologyEx(filled, cv2.MORPH_OPEN, k_big, iterations=open_iter)
    return filled


def keep_main_and_left(binary):
    if not keep_left_substrate:
        return binary
    h, w = binary.shape
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num <= 1:
        return binary
    # 主體
    areas = stats[1:, cv2.CC_STAT_AREA]
    main_id = 1 + np.argmax(areas)
    keep = {main_id}
    # 左側細長基板
    for i in range(1, num):
        if i == main_id:
            continue
        x, y, ww, hh, area = (
            stats[i, 0],
            stats[i, 1],
            stats[i, 2],
            stats[i, 3],
            stats[i, cv2.CC_STAT_AREA],
        )
        touches_left = x == 0
        tall_thin = (hh >= 0.40 * h) and (ww <= 0.15 * w)
        in_left = (x + ww) <= int(0.30 * w)
        big_enough = area >= 0.005 * (h * w)
        if touches_left and tall_thin and in_left and big_enough:
            keep.add(i)
    mask = np.isin(labels, list(keep))
    mask = binary_fill_holes(mask)
    return (mask.astype(np.uint8)) * 255


def preprocess_final(gray, filename):
    b0 = prelim_binary(gray)
    px_um = estimate_px_size_um(gray, filename)

    def run_once(basis_binary, k, c_iter, o_iter):
        if px_um is None:
            h, w = basis_binary.shape
            area = h * w
            fg = basis_binary.astype(bool)
            fg = remove_small_objects(
                fg, min_size=max(1, int(fallback_min_obj_area_ratio * area))
            )
            fg = remove_small_holes(
                fg, area_threshold=int(0.25 * fallback_max_hole_area_ratio * area)
            )
            b_small = (fg.astype(np.uint8)) * 255

            im_ff = b_small.copy()
            mask = np.zeros((h + 2, w + 2), np.uint8)
            cv2.floodFill(im_ff, mask, (0, 0), 255)
            holes = cv2.bitwise_not(im_ff)

            num, labels = cv2.connectedComponents(holes, connectivity=8)
            a_max = int(fallback_max_hole_area_ratio * area)
            holes_keep = np.zeros_like(b_small)
            for i in range(1, num):
                if (labels == i).sum() <= a_max:
                    holes_keep[labels == i] = 255
            filled = cv2.bitwise_or(b_small, holes_keep)
        else:
            filled = size_aware_fill(basis_binary, px_um)
        return filled

    k_used = kernel_size
    close_used = close_iter
    open_used = open_iter
    binary = run_once(b0, k_used, close_used, open_used)
    comp = count_components(binary, min_component_area_ratio)

    tries = 0
    while comp > max_components and tries < max_retries:
        tries += 1
        close_used += close_step
        binary = run_once(b0, k_used, close_used, open_used)
        comp = count_components(binary, min_component_area_ratio)

    if keep_left_substrate:
        binary = keep_main_and_left(binary)

    meta = {
        "px_um": px_um,
        "components": comp,
        "close_iter_used": close_used,
        "kernel_used": k_used,
    }
    return binary, px_um, meta


def classify_dir(angle_deg, th=20.0):
    if angle_deg > th:
        return "Up"
    if angle_deg < -th:
        return "Down"
    return "Flat"


def extend_line(p1, p2, scale):
    p1 = np.array(p1, dtype=float)
    p2 = np.array(p2, dtype=float)
    v = p2 - p1
    n = np.linalg.norm(v)
    if n == 0:
        return p1, p2
    v = v / n
    return p1 - v * scale, p2 + v * scale


# --------- 新增：把相鄰且同向、角度相近的斜邊合併，並用合併後的總長挑主斜邊 ----------
def merge_collinear_runs(segments, angle_range=(25.0, 80.0), ang_tol=10.0):
    """
    依照 RDP 產生的順序，把連續、同向(正/負角)且角度相近(±ang_tol)、
    且絕對角度落在 angle_range 的段落合併成長段。
    回傳的每一段含 p1, p2, angle(長度加權平均), length(總長), sgn(+1/-1)
    """

    def sgn(a):
        return 1 if a > 0 else (-1 if a < 0 else 0)

    runs = []
    cur = None

    for s in segments:
        ang = float(s["angle"])
        if not (angle_range[0] <= abs(ang) <= angle_range[1]):
            # 角度不在斜邊範圍，結束目前 run
            if cur is not None:
                runs.append(cur)
                cur = None
            continue

        this_sgn = sgn(ang)
        if cur is None:
            cur = {
                "p1": np.array(s["p1"], float),
                "p2": np.array(s["p2"], float),
                "angle": ang,
                "length": float(s["length"]),
                "sgn": this_sgn,
                "_w": float(s["length"]),
            }
            continue

        # 可合併：同向且角度接近
        if this_sgn == cur["sgn"] and abs(ang - cur["angle"]) <= ang_tol:
            cur["p2"] = np.array(s["p2"], float)
            cur["length"] += float(s["length"])
            cur["_w"] += float(s["length"])
            # 長度加權平均角度
            cur["angle"] = (
                cur["angle"] * (cur["_w"] - s["length"]) + ang * s["length"]
            ) / cur["_w"]
        else:
            runs.append(cur)
            cur = {
                "p1": np.array(s["p1"], float),
                "p2": np.array(s["p2"], float),
                "angle": ang,
                "length": float(s["length"]),
                "sgn": this_sgn,
                "_w": float(s["length"]),
            }

    if cur is not None:
        runs.append(cur)
    for r in runs:
        r.pop("_w", None)
    return runs


def select_major_slopes(segments, angle_range=(25.0, 80.0), ang_tol=10.0):
    """
    先合併，再用『合併後總長』選 Upper/Lower。
    只在 angle_range 內挑選，避免把近垂直/近水平段當主斜邊。
    """
    runs = merge_collinear_runs(segments, angle_range=angle_range, ang_tol=ang_tol)
    up = max(
        (r for r in runs if r["sgn"] == +1), key=lambda r: r["length"], default=None
    )
    dn = max(
        (r for r in runs if r["sgn"] == -1), key=lambda r: r["length"], default=None
    )
    return up, dn


# ------------------------------------------------------------------------


# ======================= 單張流程 =======================
def process_one_image(img_path, save_dir):
    name = os.path.splitext(os.path.basename(img_path))[0]
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return {"file": name, "status": "read_fail"}

    binary, px_um, meta = preprocess_final(gray, name)

    if meta.get("components", 0) <= 1 and not is_valid_main_object(binary):
        if debug_binary:
            cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)
        return {
            "file": name,
            "status": "noise_or_too_small",
            "px_um": None if px_um is None else round(float(px_um), 6),
            "n_segments": "",
            "upper_angle_deg": "",
            "lower_angle_deg": "",
            "upper_length_px": "",
            "lower_length_px": "",
            "components": meta.get("components", ""),
            "close_iter_used": meta.get("close_iter_used", ""),
            "kernel_used": meta.get("kernel_used", ""),
            "result_image": "",
        }

    if debug_binary:
        cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)

    cnt = find_main_contour(binary)
    if cnt is None:
        return {"file": name, "status": "no_contour"}

    contour = cnt.reshape(-1, 2).astype(np.float32)
    corners = approximate_polygon(contour, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])

    fitted_segments = []
    for i in range(len(corners) - 1):
        p0, p1 = corners[i], corners[i + 1]
        i0 = nearest_idx(contour, p0)
        i1 = nearest_idx(contour, p1)
        seg_pts = slice_closed_contour(contour, i0, i1)
        res = fit_segment(seg_pts)
        if res is None:
            continue
        q1, q2, ang, L = res
        fitted_segments.append(
            {
                "p1": q1,
                "p2": q2,
                "angle": ang,
                "length": L,
                "direction": classify_dir(ang, direction_thresh_deg),
            }
        )

    # ★ 改用「合併後的實際斜邊總長」來挑主斜邊
    upper, lower = select_major_slopes(
        fitted_segments, angle_range=(25.0, 80.0), ang_tol=10.0
    )

    # 視覺化
    fig, ax = plt.subplots(figsize=(6, 10))
    ax.imshow(binary, cmap="gray", origin="upper")
    ax.plot(contour[:, 0], contour[:, 1], "g-", alpha=0.25, label="Raw Contour")
    #ax.plot(corners[:, 0], corners[:, 1], "ro-", markersize=3, label="Corner Points")

    for seg in fitted_segments:
        color = (
            "blue"
            if seg["direction"] == "Up"
            else ("orange" if seg["direction"] == "Down" else "gray")
        )
        e1, e2 = extend_line(seg["p1"], seg["p2"], extend_scale)
        ax.plot(
            [e1[0], e2[0]], [e1[1], e2[1]], linestyle="--", linewidth=1.6, color=color
        )

    if upper:
        ax.plot(
            [upper["p1"][0], upper["p2"][0]],
            [upper["p1"][1], upper["p2"][1]],
            "r-",
            linewidth=2.5,
        )
        mid = (np.array(upper["p1"]) + np.array(upper["p2"])) / 2
        ax.text(mid[0], mid[1], f"Upper {upper['angle']:.1f}°", color="red", fontsize=9)

    if lower:
        ax.plot(
            [lower["p1"][0], lower["p2"][0]],
            [lower["p1"][1], lower["p2"][1]],
            "lime",
            linewidth=2.5,
        )
        mid = (np.array(lower["p1"]) + np.array(lower["p2"])) / 2
        ax.text(
            mid[0], mid[1], f"Lower {lower['angle']:.1f}°", color="green", fontsize=9
        )

    ax.text(
        0.02,
        0.98,
        f"comp={meta.get('components','?')}, close={meta.get('close_iter_used','?')}, k={meta.get('kernel_used','?')}",
        color="cyan",
        fontsize=10,
        bbox=dict(facecolor="gray", alpha=0.3),
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    ax.set_title(name)
    h, w = binary.shape
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)  # 座標與原圖一致：左上(0,0)，y 向下
    ax.set_aspect("equal")
    ax.legend(loc="upper left")
    plt.tight_layout()

    out_png = os.path.join(save_dir, f"{name}_fitted_extended.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    return {
        "file": name,
        "status": "ok",
        "px_um": None if px_um is None else round(float(px_um), 6),
        "n_segments": len(fitted_segments),
        "upper_angle_deg": None if not upper else round(float(upper["angle"]), 3),
        "lower_angle_deg": None if not lower else round(float(lower["angle"]), 3),
        "upper_length_px": None if not upper else round(float(upper["length"]), 3),
        "lower_length_px": None if not lower else round(float(lower["length"]), 3),
        "components": meta.get("components", ""),
        "close_iter_used": meta.get("close_iter_used", ""),
        "kernel_used": meta.get("kernel_used", ""),
        "result_image": out_png,
    }


# ======================= 批次流程 =======================
def batch_process(input_dir, save_dir, csv_out):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = [f for f in os.listdir(input_dir) if os.path.splitext(f.lower())[1] in exts]
    files.sort()
    if not files:
        print("⚠️ 找不到圖片檔")
        return

    rows = []
    for i, fname in enumerate(files, 1):
        path = os.path.join(input_dir, fname)
        print(f"[{i}/{len(files)}] {fname} …")
        try:
            res = process_one_image(path, save_dir)
        except Exception as e:
            res = {"file": os.path.splitext(fname)[0], "status": f"error: {e}"}
        rows.append(res)

    fieldnames = [
        "file",
        "status",
        "px_um",
        "n_segments",
        "upper_angle_deg",
        "lower_angle_deg",
        "upper_length_px",
        "lower_length_px",
        "components",
        "close_iter_used",
        "kernel_used",
        "result_image",
    ]
    with open(csv_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fieldnames})

    print(f"✅ 完成！共處理 {len(files)} 張圖。")
    print(f"📄 CSV：{csv_out}")
    print(f"🖼 圖片輸出資料夾：{save_dir}")
    if debug_binary:
        print(f"🧪 Debug 二值化輸出：{debug_dir}")


# ======================= 執行 =======================
if __name__ == "__main__":
    batch_process(input_folder, output_folder, csv_path)

[1/33] 0.3_0.3_30_01_RB(1024).png …
[2/33] 0.3_0.3_30_02_RB(1024).png …
[3/33] 0.3_0.3_60_01_RB(1024).png …
[4/33] 0.3_0.3_60_02_RB(1024).png …
[5/33] 0.3_0.3_90_01_RB(1024).png …
[6/33] 0.3_0.3_90_02_RB(1024).png …
[7/33] 0.3_0.6_30_01_RB(1024).png …
[8/33] 0.3_0.6_60_01_RB(1024).png …
[9/33] 0.3_0.6_60_02_RB(1024).png …
[10/33] 0.3_0.6_90_01_RB(1024).png …
[11/33] 0.3_0.9_30_02_RB(1024).png …
[12/33] 0.3_0.9_60_01_RB(1024).png …
[13/33] 0.3_0.9_90_01_RB(1024).png …
[14/33] 0.6_0.3_30_01_RB(1024).png …
[15/33] 0.6_0.3_60_01_RB(1024).png …
[16/33] 0.6_0.3_60_02_RB(1024).png …
[17/33] 0.6_0.3_90_01_RB(1024).png …
[18/33] 0.6_0.3_90_02_RB(1024).png …
[19/33] 0.6_0.6_30_01_RB(1024).png …
[20/33] 0.6_0.6_60_01_RB(1024).png …
[21/33] 0.6_0.6_90_01_RB(1024).png …
[22/33] 0.6_0.9_30_02_RB(1024).png …
[23/33] 0.6_0.9_60_01_RB(1024).png …
[24/33] 0.6_0.9_90_01_RB(1024).png …
[25/33] 0.9_0.3_30_01_RB(1024).png …
[26/33] 0.9_0.3_60_06_RB(1024).png …
[27/33] 0.9_0.3_90_01_RB(1024).png …
[28/33] 0.

In [59]:
# -*- coding: utf-8 -*-
import os
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt
from math import degrees
from skimage.measure import approximate_polygon
from sklearn.linear_model import LinearRegression, RANSACRegressor
from scipy.ndimage import binary_fill_holes

# ======================= 全域參數 =======================
# 幾何擬合 / 顯示
RDP_TOLERANCE = 10  # 調小會切更多段（藍色虛線更多）
EXTEND_SCALE = 120.0

# 形態學（kernel 固定，不自動放大）
KERNEL_SIZE = 7
CLOSE_ITER = 2
OPEN_ITER = 1

# 連通域自動調參（只增 close、不增 kernel）
MAX_COMPONENTS = 1
MIN_COMPONENT_AREA_RATIO = 0.001
MAX_RETRIES = 50
CLOSE_STEP = 6

# 主體有效性門檻（避免 components==1 但其實是雜訊）
MIN_MAIN_AREA_RATIO = 0.02
MIN_MAIN_SHORT_SIDE = 80

# 是否保留左側細長基板（預設 False）
KEEP_LEFT_SUBSTRATE = False

# I/O （自行更改）
input_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB"
output_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0810ResultFigures"
os.makedirs(output_folder, exist_ok=True)
csv_path = os.path.join(output_folder, "batch_edge_summary.csv")

# Debug 二值圖
DEBUG_BINARY = True
debug_dir = os.path.join(output_folder, "_debug_binary")
if DEBUG_BINARY:
    os.makedirs(debug_dir, exist_ok=True)


# ======================= 小工具 =======================
def prelim_binary(gray):
    """穩定簡單版：Otsu + 小型態學"""
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k, iterations=2)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, k, iterations=1)
    return b


def count_components(binary, min_area_ratio=0.001):
    h, w = binary.shape
    area = h * w
    min_area = max(1, int(min_area_ratio * area))
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    return sum(int(stats[i, cv2.CC_STAT_AREA] >= min_area) for i in range(1, num))


def is_valid_main_object(binary):
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num <= 1:
        return False
    idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    h, w = binary.shape
    img_area = h * w
    main_area = stats[idx, cv2.CC_STAT_AREA]
    bw = stats[idx, cv2.CC_STAT_WIDTH]
    bh = stats[idx, cv2.CC_STAT_HEIGHT]
    return (main_area / img_area) >= MIN_MAIN_AREA_RATIO and min(
        int(bw), int(bh)
    ) >= int(MIN_MAIN_SHORT_SIDE)


def keep_main_and_left(binary):
    """（可選）保留最大主體 + 左側細長基板"""
    if not KEEP_LEFT_SUBSTRATE:
        return binary
    h, w = binary.shape
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num <= 1:
        return binary
    areas = stats[1:, cv2.CC_STAT_AREA]
    main_id = 1 + np.argmax(areas)
    keep = {main_id}
    for i in range(1, num):
        if i == main_id:
            continue
        x, y, ww, hh, area = (
            stats[i, 0],
            stats[i, 1],
            stats[i, 2],
            stats[i, 3],
            stats[i, cv2.CC_STAT_AREA],
        )
        touches_left = x == 0
        tall_thin = (hh >= 0.40 * h) and (ww <= 0.15 * w)
        in_left = (x + ww) <= int(0.30 * w)
        big_enough = area >= 0.005 * (h * w)
        if touches_left and tall_thin and in_left and big_enough:
            keep.add(i)
    mask = np.isin(labels, list(keep))
    mask = binary_fill_holes(mask)
    return (mask.astype(np.uint8)) * 255


def find_main_contour(binary):
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return max(cnts, key=cv2.contourArea) if cnts else None


def nearest_idx(contour, pt):
    d = np.linalg.norm(contour - pt, axis=1)
    return int(np.argmin(d))


def slice_closed_contour(contour, i0, i1):
    return (
        contour[i0 : i1 + 1]
        if i0 <= i1
        else np.vstack([contour[i0:], contour[: i1 + 1]])
    )


def fit_segment(points_xy):
    """用線性回歸擬合一段，回傳端點、角度（右=0°, 上=正）、視覺長度"""
    if len(points_xy) < 2:
        return None
    X = points_xy[:, 0].reshape(-1, 1)
    y = points_xy[:, 1]
    reg = LinearRegression().fit(X, y)
    x1, x2 = float(X[0, 0]), float(X[-1, 0])
    y1 = float(reg.predict([[x1]])[0])
    y2 = float(reg.predict([[x2]])[0])
    ang = degrees(np.arctan2(-(y2 - y1), (x2 - x1)))  # 右=0°, 順時針為負（影像 y 向下）
    L = float(np.hypot(x2 - x1, y2 - y1))
    return (x1, y1), (x2, y2), ang, L


def rdp_fit_segments(contour_xy, rdp_tolerance=15.0):
    """RDP 取角點 → 相鄰角點之間各自線性擬合成段"""
    corners = approximate_polygon(contour_xy, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])
    segs = []
    for i in range(len(corners) - 1):
        i0 = nearest_idx(contour_xy, corners[i])
        i1 = nearest_idx(contour_xy, corners[i + 1])
        pts = slice_closed_contour(contour_xy, i0, i1)
        res = fit_segment(pts)
        if res is not None:
            p1, p2, ang, L = res
            segs.append({"p1": p1, "p2": p2, "angle": ang, "length": L})
    return segs, corners


def extend_line(p1, p2, scale):
    p1 = np.array(p1, dtype=float)
    p2 = np.array(p2, dtype=float)
    v = p2 - p1
    n = np.linalg.norm(v)
    if n == 0:
        return p1, p2
    v = v / n
    return p1 - v * scale, p2 + v * scale


def norm_tilt(angle_deg):
    """相對『水平』的傾斜角度，0~90（避免 ±180 的模糊）"""
    a = abs(angle_deg) % 180.0
    return a if a <= 90.0 else 180.0 - a


def angle_diff(a, b):
    """兩角度的夾角(0~90°)"""
    d = abs((a - b) % 180.0)
    return d if d <= 90.0 else 180.0 - d


# --------- 斜邊就近配對（可得到多節）---------
def pair_slopes_nearest(
    segments,
    slope_min_deg=25,
    slope_max_deg=80,
    join_eps_px=60,
    min_len_px=120,
    min_y_ovlp_ratio=0.10,
):
    """
    不分 Up/Down；直接用 斜度+長度+端點距離+垂直重疊 來就近配對兩條斜邊
    回傳 [(upper_seg, lower_seg), ...]（upper=角度較大者，僅為配色一致）
    """

    def is_slope(seg):
        t = norm_tilt(seg["angle"])
        return (slope_min_deg <= t <= slope_max_deg) and (seg["length"] >= min_len_px)

    cand = [s for s in segments if is_slope(s)]

    def cy(s):  # 依中心 y 排序
        return 0.5 * (s["p1"][1] + s["p2"][1])

    cand.sort(key=cy)

    used = set()
    pairs = []

    def min_endpt_dist(a, b):
        ax = [a["p1"][0], a["p2"][0]]
        ay = [a["p1"][1], a["p2"][1]]
        bx = [b["p1"][0], b["p2"][0]]
        by = [b["p1"][1], b["p2"][1]]
        d = []
        for i in (0, 1):
            for j in (0, 1):
                d.append(np.hypot(ax[i] - bx[j], ay[i] - by[j]))
        return min(d)

    def y_overlap(a, b):
        ay0, ay1 = sorted([a["p1"][1], a["p2"][1]])
        by0, by1 = sorted([b["p1"][1], b["p2"][1]])
        return max(0.0, min(ay1, by1) - max(ay0, by0))

    for i, s in enumerate(cand):
        if i in used:
            continue
        best = None
        best_key = None
        for j in range(i + 1, len(cand)):
            if j in used:
                continue
            t = cand[j]
            d = min_endpt_dist(s, t)
            if d > join_eps_px:
                continue
            ov = y_overlap(s, t)
            min_h = min(abs(s["p2"][1] - s["p1"][1]), abs(t["p2"][1] - t["p1"][1]))
            if min_h <= 0 or ov < min_y_ovlp_ratio * min_h:
                continue
            key = (d, -(s["length"] * t["length"]))
            if best is None or key < best_key:
                best = (j, t)
                best_key = key

        if best is not None:
            j, t = best
            used.add(i)
            used.add(j)
            up, down = (s, t) if s["angle"] >= t["angle"] else (t, s)
            pairs.append((up, down))
    return pairs


# --------- 左牆擬合（逐列最左白點 → x=a*y+b）---------
def fit_left_wall_from_mask(binary, y_step=4, trim_ratio=0.08):
    """
    逐列(y)找最左邊前景像素 -> 擬合直線。
    回傳: (p1, p2, angle) 或 None
    """
    h, w = binary.shape
    ys = np.arange(0, h, y_step, dtype=int)

    xs, ysel = [], []
    for y in ys:
        row = binary[y] > 0
        if row.any():
            xs.append(int(np.argmax(row)))
            ysel.append(y)

    if len(xs) < max(30, h // 200):
        return None

    X = np.array(ysel, dtype=float).reshape(-1, 1)  # y
    Y = np.array(xs, dtype=float)  # x
    reg = LinearRegression().fit(X, Y)
    resid = np.abs(Y - reg.predict(X))
    thr = np.quantile(resid, 1 - trim_ratio)
    keep = resid <= thr
    if keep.sum() >= 20:
        reg = LinearRegression().fit(X[keep], Y[keep])
        X = X[keep]

    y1, y2 = float(X.min()), float(X.max())
    x1 = float(reg.predict([[y1]])[0])
    x2 = float(reg.predict([[y2]])[0])
    ang = degrees(np.arctan2(-(y2 - y1), (x2 - x1)))  # 與其他角度定義一致
    return (x1, y1), (x2, y2), ang


# ======================= 前處理（含自動 close 疊加） =======================
def preprocess_final(gray):
    b0 = prelim_binary(gray)

    def run_once(basis_binary, k, c_iter, o_iter):
        k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        filled = cv2.morphologyEx(
            basis_binary, cv2.MORPH_CLOSE, k_big, iterations=c_iter
        )
        filled = cv2.morphologyEx(filled, cv2.MORPH_OPEN, k_big, iterations=o_iter)
        return filled

    k_used = KERNEL_SIZE
    close_used = CLOSE_ITER
    open_used = OPEN_ITER

    binary = run_once(b0, k_used, close_used, open_used)
    comp = count_components(binary, MIN_COMPONENT_AREA_RATIO)

    tries = 0
    while comp > MAX_COMPONENTS and tries < MAX_RETRIES:
        tries += 1
        close_used += CLOSE_STEP
        binary = run_once(b0, k_used, close_used, open_used)
        comp = count_components(binary, MIN_COMPONENT_AREA_RATIO)

    if KEEP_LEFT_SUBSTRATE:
        binary = keep_main_and_left(binary)

    meta = {"components": comp, "close_iter_used": close_used, "kernel_used": k_used}
    return binary, meta


# ======================= 單張流程 =======================
def process_one_image(img_path, save_dir):
    name = os.path.splitext(os.path.basename(img_path))[0]
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return {"file": name, "status": "read_fail"}

    binary, meta = preprocess_final(gray)

    # 主體檢查
    if meta.get("components", 0) <= 1 and not is_valid_main_object(binary):
        if DEBUG_BINARY:
            cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)
        return {
            "file": name,
            "status": "noise_or_too_small",
            "n_pairs": 0,
            "pairs_angles_deg": "",
            "pairs_angles_vs_left_deg": "",
            "left_wall_angle_deg": "",
            "components": meta.get("components", ""),
            "close_iter_used": meta.get("close_iter_used", ""),
            "kernel_used": meta.get("kernel_used", ""),
            "result_image": "",
        }

    if DEBUG_BINARY:
        cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)

    # 輪廓 → 角點 → 各段擬合
    cnt = find_main_contour(binary)
    if cnt is None:
        return {"file": name, "status": "no_contour"}

    contour = cnt.reshape(-1, 2).astype(np.float32)
    segments, corners = rdp_fit_segments(contour, RDP_TOLERANCE)

    # 依影像尺寸自動選擇成對門檻
    h, w = binary.shape
    PAIR_SLOPE_MIN_DEG = 20
    PAIR_SLOPE_MAX_DEG = 82
    PAIR_JOIN_EPS_PX = max(50, int(0.06 * w))
    PAIR_MIN_LEN_PX = max(50, int(0.10 * h))
    PAIR_MIN_YOVLP_RATIO = 0.08

    pairs = pair_slopes_nearest(
        segments,
        slope_min_deg=PAIR_SLOPE_MIN_DEG,
        slope_max_deg=PAIR_SLOPE_MAX_DEG,
        join_eps_px=PAIR_JOIN_EPS_PX,
        min_len_px=PAIR_MIN_LEN_PX,
        min_y_ovlp_ratio=PAIR_MIN_YOVLP_RATIO,
    )
    # 容錯：放寬一次門檻
    if not pairs:
        pairs = pair_slopes_nearest(
            segments,
            slope_min_deg=max(10, PAIR_SLOPE_MIN_DEG - 10),
            slope_max_deg=min(88, PAIR_SLOPE_MAX_DEG + 8),
            join_eps_px=int(PAIR_JOIN_EPS_PX * 1.6),
            min_len_px=int(PAIR_MIN_LEN_PX * 0.6),
            min_y_ovlp_ratio=max(0.03, PAIR_MIN_YOVLP_RATIO * 0.5),
        )

    # 視覺化
    fig, ax = plt.subplots(figsize=(6.5, 10))
    ax.imshow(binary, cmap="gray", origin="upper")
    # 原始輪廓（淡綠）
    ax.plot(
        contour[:, 0],
        contour[:, 1],
        color=(0.3, 0.8, 0.6, 0.4),
        linewidth=1.0,
        label="Raw Contour",
    )
    # 角點
    ax.scatter(
        corners[:, 0],
        corners[:, 1],
        s=10,
        c="red",
        edgecolors="white",
        linewidths=0.5,
        label="Corner Points",
    )

    # 擬合左牆
    left_line = fit_left_wall_from_mask(binary, y_step=5, trim_ratio=0.08)
    left_ang = None
    if left_line is not None:
        (lx1, ly1), (lx2, ly2), left_ang = left_line
        (lx1e, ly1e), (lx2e, ly2e) = extend_line(
            (lx1, ly1), (lx2, ly2), EXTEND_SCALE * 2
        )
        ax.plot(
            [lx1e, lx2e], [ly1e, ly2e], color="gold", linewidth=3, label="Left Wall"
        )
        mx, my = (lx1 + lx2) / 2.0, (ly1 + ly2) / 2.0
        ax.text(mx, my, f"Left {left_ang:.1f}°", color="gold", fontsize=11)

    # 逐組結構畫兩斜邊，並計算各自與左牆夾角
    pair_strs = []
    pair_vs_left = []
    for upper, lower in pairs:
        u1, u2 = extend_line(upper["p1"], upper["p2"], EXTEND_SCALE)
        l1, l2 = extend_line(lower["p1"], lower["p2"], EXTEND_SCALE)
        ax.plot([u1[0], u2[0]], [u1[1], u2[1]], "r-", linewidth=2.2)
        ax.plot([l1[0], l2[0]], [l1[1], l2[1]], color="lime", linewidth=2.2)

        um = (np.array(upper["p1"]) + np.array(upper["p2"])) / 2.0
        lm = (np.array(lower["p1"]) + np.array(lower["p2"])) / 2.0

        # 舊：原角度（若想同時顯示可解註）
        # ax.text(um[0], um[1], f"Upper {upper['angle']:.1f}°", color="red",  fontsize=9)
        # ax.text(lm[0], lm[1], f"Lower {lower['angle']:.1f}°", color="lime", fontsize=9)

        if left_ang is not None:
            du = angle_diff(upper["angle"], left_ang)
            dl = angle_diff(lower["angle"], left_ang)
            ax.text(um[0], um[1] - 25, f"∠{du:.1f}°", color="yellow", fontsize=10)
            ax.text(lm[0], lm[1] - 25, f"∠{dl:.1f}°", color="greenyellow", fontsize=10)
            pair_vs_left.append(f"{du:.1f}/{dl:.1f}")

        pair_strs.append(f"{upper['angle']:.1f}/{lower['angle']:.1f}")

    # 左上角小標籤：comp/close/kernel
    ax.text(
        0.02,
        0.98,
        f"comp={meta.get('components','?')}, close={meta.get('close_iter_used','?')}, k={meta.get('kernel_used','?')}",
        color="cyan",
        fontsize=10,
        bbox=dict(facecolor="gray", alpha=0.3),
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    ax.set_title(name)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)  # y 向下
    ax.set_aspect("equal")
    ax.legend(loc="upper left")
    plt.tight_layout()

    out_png = os.path.join(save_dir, f"{name}_fitted_extended.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    # 取全圖「最強一組」作為舊欄位輸出（長度較短的那條越大越好）
    if pairs:
        best_pair = max(pairs, key=lambda pr: min(pr[0]["length"], pr[1]["length"]))
        best_u, best_l = best_pair
        best_u_ang = round(float(best_u["angle"]), 3)
        best_l_ang = round(float(best_l["angle"]), 3)
    else:
        best_u_ang = ""
        best_l_ang = ""

    return {
        "file": name,
        "status": "ok",
        "n_pairs": len(pairs),
        "pairs_angles_deg": ";".join(pair_strs),  # "up/down;up/down;..."
        "pairs_angles_vs_left_deg": ";".join(pair_vs_left) if pair_vs_left else "",
        "left_wall_angle_deg": (
            round(float(left_ang), 3) if left_ang is not None else ""
        ),
        "upper_angle_deg": best_u_ang,
        "lower_angle_deg": best_l_ang,
        "components": meta.get("components", ""),
        "close_iter_used": meta.get("close_iter_used", ""),
        "kernel_used": meta.get("kernel_used", ""),
        "result_image": out_png,
    }


# ======================= 批次流程 =======================
def batch_process(input_dir, save_dir, csv_out):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = [f for f in os.listdir(input_dir) if os.path.splitext(f.lower())[1] in exts]
    files.sort()
    if not files:
        print("⚠️ 找不到圖片檔")
        return

    rows = []
    for i, fname in enumerate(files, 1):
        path = os.path.join(input_dir, fname)
        print(f"[{i}/{len(files)}] {fname} …")
        try:
            res = process_one_image(path, save_dir)
        except Exception as e:
            res = {"file": os.path.splitext(fname)[0], "status": f"error: {e}"}
        rows.append(res)

    fieldnames = [
        "file",
        "status",
        "n_pairs",
        "pairs_angles_deg",
        "pairs_angles_vs_left_deg",
        "left_wall_angle_deg",
        "upper_angle_deg",
        "lower_angle_deg",
        "components",
        "close_iter_used",
        "kernel_used",
        "result_image",
    ]
    with open(csv_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fieldnames})

    print(f"✅ 完成！共處理 {len(files)} 張圖。")
    print(f"📄 CSV：{csv_out}")
    print(f"🖼 圖片輸出資料夾：{save_dir}")
    if DEBUG_BINARY:
        print(f"🧪 Debug 二值化輸出：{debug_dir}")


# ======================= 執行 =======================
if __name__ == "__main__":
    batch_process(input_folder, output_folder, csv_path)

[1/33] 0.3_0.3_30_01_RB(1024).png …
[2/33] 0.3_0.3_30_02_RB(1024).png …
[3/33] 0.3_0.3_60_01_RB(1024).png …
[4/33] 0.3_0.3_60_02_RB(1024).png …
[5/33] 0.3_0.3_90_01_RB(1024).png …
[6/33] 0.3_0.3_90_02_RB(1024).png …
[7/33] 0.3_0.6_30_01_RB(1024).png …
[8/33] 0.3_0.6_60_01_RB(1024).png …
[9/33] 0.3_0.6_60_02_RB(1024).png …
[10/33] 0.3_0.6_90_01_RB(1024).png …
[11/33] 0.3_0.9_30_02_RB(1024).png …
[12/33] 0.3_0.9_60_01_RB(1024).png …
[13/33] 0.3_0.9_90_01_RB(1024).png …
[14/33] 0.6_0.3_30_01_RB(1024).png …
[15/33] 0.6_0.3_60_01_RB(1024).png …
[16/33] 0.6_0.3_60_02_RB(1024).png …
[17/33] 0.6_0.3_90_01_RB(1024).png …
[18/33] 0.6_0.3_90_02_RB(1024).png …
[19/33] 0.6_0.6_30_01_RB(1024).png …
[20/33] 0.6_0.6_60_01_RB(1024).png …
[21/33] 0.6_0.6_90_01_RB(1024).png …
[22/33] 0.6_0.9_30_02_RB(1024).png …
[23/33] 0.6_0.9_60_01_RB(1024).png …
[24/33] 0.6_0.9_90_01_RB(1024).png …
[25/33] 0.9_0.3_30_01_RB(1024).png …
[26/33] 0.9_0.3_60_06_RB(1024).png …
[27/33] 0.9_0.3_90_01_RB(1024).png …
[28/33] 0.

## 0811

In [1]:
# -*- coding: utf-8 -*-
"""
RB edge analyzer (integrated)
- Keep ALL right-side sloped segments (no merge)
- Robustly fit the left wall
- For EVERY sloped segment, compute angle vs left wall
- Save visualization and CSV
"""

import os
import csv
import cv2
import numpy as np
import matplotlib.pyplot as plt
from math import degrees
from skimage.measure import approximate_polygon
from sklearn.linear_model import LinearRegression
from scipy.ndimage import binary_fill_holes

# ======================= 全域參數 =======================
# 幾何擬合 / 顯示
RDP_TOLERANCE = 11.0  # 小→角點較多→斜邊切得更細
EXTEND_SCALE = 110.0

# 形態學（kernel 固定，不自動放大）
KERNEL_SIZE = 7
CLOSE_ITER = 2
OPEN_ITER = 1

# 連通域自動調參（只增 close，不增 kernel）
MAX_COMPONENTS = 1
MIN_COMPONENT_AREA_RATIO = 0.001
MAX_RETRIES = 50
CLOSE_STEP = 6

# 主體有效性門檻（避免 components==1 但其實是雜訊）
MIN_MAIN_AREA_RATIO = 0.02
MIN_MAIN_SHORT_SIDE = 80

# 是否保留左側細長基板（預設 False）
KEEP_LEFT_SUBSTRATE = False

# I/O
input_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0.6_0.9"
output_folder = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0.6_0.9\0810ResultFigures"
os.makedirs(output_folder, exist_ok=True)
csv_path = os.path.join(output_folder, "batch_edge_summary.csv")

# Debug 二值圖
DEBUG_BINARY = True
debug_dir = os.path.join(output_folder, "_debug_binary")
if DEBUG_BINARY:
    os.makedirs(debug_dir, exist_ok=True)


# ======================= 小工具 =======================
def prelim_binary(gray):
    """穩定簡單版：Otsu + 小型態學"""
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k, iterations=2)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, k, iterations=1)
    return b


def count_components(binary, min_area_ratio=0.001):
    h, w = binary.shape
    area = h * w
    min_area = max(1, int(min_area_ratio * area))
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    return sum(int(stats[i, cv2.CC_STAT_AREA] >= min_area) for i in range(1, num))


def is_valid_main_object(binary):
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num <= 1:
        return False
    idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    h, w = binary.shape
    img_area = h * w
    main_area = stats[idx, cv2.CC_STAT_AREA]
    bw = stats[idx, cv2.CC_STAT_WIDTH]
    bh = stats[idx, cv2.CC_STAT_HEIGHT]
    return (main_area / img_area) >= MIN_MAIN_AREA_RATIO and min(
        int(bw), int(bh)
    ) >= int(MIN_MAIN_SHORT_SIDE)


def keep_main_and_left(binary):
    """（可選）保留最大主體 + 左側細長基板"""
    if not KEEP_LEFT_SUBSTRATE:
        return binary
    h, w = binary.shape
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num <= 1:
        return binary
    areas = stats[1:, cv2.CC_STAT_AREA]
    main_id = 1 + np.argmax(areas)
    keep = {main_id}
    for i in range(1, num):
        if i == main_id:
            continue
        x, y, ww, hh, area = (
            stats[i, 0],
            stats[i, 1],
            stats[i, 2],
            stats[i, 3],
            stats[i, cv2.CC_STAT_AREA],
        )
        touches_left = x == 0
        tall_thin = (hh >= 0.40 * h) and (ww <= 0.15 * w)
        in_left = (x + ww) <= int(0.30 * w)
        big_enough = area >= 0.005 * (h * w)
        if touches_left and tall_thin and in_left and big_enough:
            keep.add(i)
    mask = np.isin(labels, list(keep))
    mask = binary_fill_holes(mask)
    return (mask.astype(np.uint8)) * 255


def find_main_contour(binary):
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return max(cnts, key=cv2.contourArea) if cnts else None


def nearest_idx(contour, pt):
    d = np.linalg.norm(contour - pt, axis=1)
    return int(np.argmin(d))


def slice_closed_contour(contour, i0, i1):
    return (
        contour[i0 : i1 + 1]
        if i0 <= i1
        else np.vstack([contour[i0:], contour[: i1 + 1]])
    )


def fit_segment(points_xy):
    """線性回歸擬合一段，回傳端點、角度（右=0°, 上=正）、視覺長度"""
    if len(points_xy) < 2:
        return None
    X = points_xy[:, 0].reshape(-1, 1)
    y = points_xy[:, 1]
    reg = LinearRegression().fit(X, y)
    x1, x2 = float(X[0, 0]), float(X[-1, 0])
    y1 = float(reg.predict([[x1]])[0])
    y2 = float(reg.predict([[x2]])[0])
    ang = degrees(np.arctan2(-(y2 - y1), (x2 - x1)))  # 右=0°, 順時針為負（y 向下）
    L = float(np.hypot(x2 - x1, y2 - y1))
    return (x1, y1), (x2, y2), ang, L


def rdp_fit_segments(contour_xy, rdp_tolerance=15.0):
    """RDP 取角點 → 相鄰角點之間各自線性擬合成段"""
    corners = approximate_polygon(contour_xy, tolerance=rdp_tolerance)
    if not np.allclose(corners[0], corners[-1]):
        corners = np.vstack([corners, corners[0]])
    segs = []
    for i in range(len(corners) - 1):
        i0 = nearest_idx(contour_xy, corners[i])
        i1 = nearest_idx(contour_xy, corners[i + 1])
        pts = slice_closed_contour(contour_xy, i0, i1)
        res = fit_segment(pts)
        if res is not None:
            p1, p2, ang, L = res
            segs.append({"p1": p1, "p2": p2, "angle": ang, "length": L})
    return segs, corners


# ---- 幾何工具 ----
def extend_line(p1, p2, scale):
    p1 = np.array(p1, dtype=float)
    p2 = np.array(p2, dtype=float)
    v = p2 - p1
    n = np.linalg.norm(v)
    if n == 0:
        return p1, p2
    v = v / n
    return p1 - v * scale, p2 + v * scale


def norm_tilt(angle_deg):
    """相對『水平』的傾斜角度，0~90（避免 ±180 的模糊）"""
    a = abs(angle_deg) % 180.0
    return a if a <= 90.0 else 180.0 - a


def angle_diff(a, b):
    """兩線夾角（0~90）"""
    d = abs((a - b) % 180.0)
    return d if d <= 90.0 else 180.0 - d


# ---- 左牆：逐列找最左白點 → 線性回歸（一次修剪） ----
def fit_left_wall_from_mask(binary, y_step=5, trim_ratio=0.08):
    h, w = binary.shape
    ys = np.arange(0, h, y_step, dtype=int)
    xs, ysel = [], []
    for y in ys:
        row = binary[y] > 0
        if row.any():
            x = int(np.argmax(row))  # 這列最左的白點
            xs.append(x)
            ysel.append(y)
    if len(xs) < max(30, h // 200):
        return None
    X = np.array(ysel, dtype=float).reshape(-1, 1)  # y → x
    Y = np.array(xs, dtype=float)
    reg = LinearRegression().fit(X, Y)
    resid = np.abs(Y - reg.predict(X))
    thr = np.quantile(resid, 1 - trim_ratio)
    keep = resid <= thr
    if keep.sum() >= 20:
        reg = LinearRegression().fit(X[keep], Y[keep])
        X = X[keep]
    y1, y2 = float(X.min()), float(X.max())
    x1 = float(reg.predict([[y1]])[0])
    x2 = float(reg.predict([[y2]])[0])
    ang = degrees(np.arctan2(-(y2 - y1), (x2 - x1)))  # 跟主角度定義一致
    return (x1, y1), (x2, y2), ang


# ---- 右側：保留所有斜邊（不合併） ----
def pick_right_slopes(
    contour_xy, segments, corners, h, angle_range=(12.0, 88.0), right_quantile=0.55
):
    """
    在右側保留所有鋸齒斜邊（不合併）：
    - 位置：x 超過 right_quantile 分位數
    - 角度：angle_range（相對水平的傾角，0~90）
    - 長度：自適應最小長度（由右側角點的垂直 pitch 推得）
    - 保底：雖短但垂直跨幅夠，也保留
    """
    # 右側位置門檻（較寬鬆）
    x_gate = float(np.quantile(contour_xy[:, 0], right_quantile))

    # 估計右側齒距的垂直 pitch -> 自適應最小長度
    rc = corners[corners[:, 0] >= x_gate]
    if len(rc) >= 4:
        dy = np.abs(np.diff(rc[:, 1]))
        dy = dy[dy > 0]
        pitch = float(np.median(dy)) if len(dy) else 0.0
    else:
        pitch = 0.0
    # 退回值：圖高的 6%
    fallback = 0.06 * h
    min_len_px = int(max(8.0, 0.3 * pitch, fallback))

    kept = []
    for s in segments:
        # 位置在右側
        cx = 0.5 * (s["p1"][0] + s["p2"][0])
        if cx < x_gate:
            continue
        # 角度（對水平的傾角）
        tilt = norm_tilt(s["angle"])
        if not (angle_range[0] <= tilt <= angle_range[1]):
            continue

        # 長度 or 垂直跨幅達標（避免因 RDP 切太細而漏掉）
        vspan = abs(s["p2"][1] - s["p1"][1])
        if (s["length"] >= min_len_px) or (vspan >= 0.6 * min_len_px):
            kept.append(s)

    return kept


# ======================= 前處理（含自動 close 疊加） =======================
def preprocess_final(gray):
    b0 = prelim_binary(gray)

    def run_once(basis_binary, k, c_iter, o_iter):
        k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        filled = cv2.morphologyEx(
            basis_binary, cv2.MORPH_CLOSE, k_big, iterations=c_iter
        )
        filled = cv2.morphologyEx(filled, cv2.MORPH_OPEN, k_big, iterations=o_iter)
        return filled

    k_used = KERNEL_SIZE
    close_used = CLOSE_ITER
    open_used = OPEN_ITER

    binary = run_once(b0, k_used, close_used, open_used)
    comp = count_components(binary, MIN_COMPONENT_AREA_RATIO)

    tries = 0
    while comp > MAX_COMPONENTS and tries < MAX_RETRIES:
        tries += 1
        close_used += CLOSE_STEP
        binary = run_once(b0, k_used, close_used, open_used)
        comp = count_components(binary, MIN_COMPONENT_AREA_RATIO)

    if KEEP_LEFT_SUBSTRATE:
        binary = keep_main_and_left(binary)

    meta = {
        "components": comp,
        "close_iter_used": close_used,
        "kernel_used": k_used,
    }
    return binary, meta


# ======================= 單張流程 =======================
def process_one_image(img_path, save_dir):
    name = os.path.splitext(os.path.basename(img_path))[0]
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return {"file": name, "status": "read_fail"}

    binary, meta = preprocess_final(gray)

    if meta.get("components", 0) <= 1 and not is_valid_main_object(binary):
        if DEBUG_BINARY:
            cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)
        return {
            "file": name,
            "status": "noise_or_too_small",
            "n_slopes": 0,
            "angles_vs_left_deg": "",
            "left_wall_angle_deg": "",
            "components": meta.get("components", ""),
            "close_iter_used": meta.get("close_iter_used", ""),
            "kernel_used": meta.get("kernel_used", ""),
            "result_image": "",
        }

    if DEBUG_BINARY:
        cv2.imwrite(os.path.join(debug_dir, f"{name}_binary.png"), binary)

    cnt = find_main_contour(binary)
    if cnt is None:
        return {"file": name, "status": "no_contour"}

    contour = cnt.reshape(-1, 2).astype(np.float32)
    segments, corners = rdp_fit_segments(contour, RDP_TOLERANCE)
    h, w = binary.shape

    # 擬合左牆（維持你目前的方法）
    left_line = fit_left_wall_from_mask(binary, y_step=5, trim_ratio=0.08)
    left_ang = None
    if left_line is not None:
        (lx1, ly1), (lx2, ly2), left_ang = left_line

    # ➜ 右側所有斜邊（使用新的自適應規則）
    all_slopes = pick_right_slopes(contour, segments, corners, h,
                                angle_range=(12, 88), right_quantile=0.52)

    angles_vs_left = []
    if left_ang is not None:
        for s in all_slopes:
            angles_vs_left.append(round(float(angle_diff(s["angle"], left_ang)), 3))

    # 視覺化
    fig, ax = plt.subplots(figsize=(7.5, 10))
    ax.imshow(binary, cmap="gray", origin="upper")
    ax.plot(
        contour[:, 0],
        contour[:, 1],
        color=(0.3, 0.8, 0.6, 0.4),
        linewidth=1.0,
        label="Raw Contour",
    )
    ax.scatter(
        corners[:, 0],
        corners[:, 1],
        s=10,
        c="red",
        edgecolors="white",
        linewidths=0.5,
        label="Corner Points",
    )

    # 左牆
    if left_ang is not None:
        (lx1e, ly1e), (lx2e, ly2e) = extend_line(
            (lx1, ly1), (lx2, ly2), EXTEND_SCALE * 2
        )
        ax.plot(
            [lx1e, lx2e], [ly1e, ly2e], color="gold", linewidth=3, label="Left Wall"
        )
        ax.text(
            (lx1 + lx2) / 2,
            (ly1 + ly2) / 2,
            f"Left {left_ang:.1f}°",
            color="gold",
            fontsize=11,
        )

    # 每一節右側斜邊 + 與左牆夾角
    for s in all_slopes:
        e1, e2 = extend_line(s["p1"], s["p2"], EXTEND_SCALE)
        ax.plot(
            [e1[0], e2[0]], [e1[1], e2[1]], linestyle="--", linewidth=1.8, color="blue"
        )
        if left_ang is not None:
            mid = (np.array(s["p1"]) + np.array(s["p2"])) / 2.0
            d = angle_diff(s["angle"], left_ang)
            ax.text(mid[0], mid[1] - 18, f"∠{d:.1f}°", color="yellow", fontsize=10)

    # 左上角小標籤：comp/close/kernel
    ax.text(
        0.02,
        0.98,
        f"comp={meta.get('components','?')}, close={meta.get('close_iter_used','?')}, k={meta.get('kernel_used','?')}",
        color="cyan",
        fontsize=10,
        bbox=dict(facecolor="gray", alpha=0.3),
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    ax.set_title(name)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.set_aspect("equal")
    ax.legend(loc="upper left")
    plt.tight_layout()

    out_png = os.path.join(save_dir, f"{name}_fitted_extended.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    return {
        "file": name,
        "status": "ok",
        "n_slopes": len(all_slopes),
        "angles_vs_left_deg": (
            ";".join(map(str, angles_vs_left)) if angles_vs_left else ""
        ),
        "left_wall_angle_deg": (
            round(float(left_ang), 3) if left_ang is not None else ""
        ),
        "components": meta.get("components", ""),
        "close_iter_used": meta.get("close_iter_used", ""),
        "kernel_used": meta.get("kernel_used", ""),
        "result_image": out_png,
    }


# ======================= 批次流程 =======================
def batch_process(input_dir, save_dir, csv_out):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    files = [f for f in os.listdir(input_dir) if os.path.splitext(f.lower())[1] in exts]
    files.sort()
    if not files:
        print("⚠️ 找不到圖片檔")
        return

    rows = []
    for i, fname in enumerate(files, 1):
        path = os.path.join(input_dir, fname)
        print(f"[{i}/{len(files)}] {fname} …")
        try:
            res = process_one_image(path, save_dir)
        except Exception as e:
            res = {"file": os.path.splitext(fname)[0], "status": f"error: {e}"}
        rows.append(res)

    fieldnames = [
        "file",
        "status",
        "n_slopes",
        "angles_vs_left_deg",
        "left_wall_angle_deg",
        "components",
        "close_iter_used",
        "kernel_used",
        "result_image",
    ]
    with open(csv_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fieldnames})

    print(f"✅ 完成！共處理 {len(files)} 張圖。")
    print(f"📄 CSV：{csv_out}")
    print(f"🖼 圖片輸出資料夾：{save_dir}")
    if DEBUG_BINARY:
        print(f"🧪 Debug 二值化輸出：{debug_dir}")


# ======================= 執行 =======================
if __name__ == "__main__":
    batch_process(input_folder, output_folder, csv_path)

[1/12] 0.6_0.6_30_01_RB(1024).png …
[2/12] 0.6_0.6_60_01_RB(1024).png …
[3/12] 0.6_0.6_90_01_RB(1024).png …
[4/12] 0.6_0.9_30_02_RB(1024).png …
[5/12] 0.6_0.9_60_01_RB(1024).png …
[6/12] 0.6_0.9_90_01_RB(1024).png …
[7/12] 0.9_0.6_30_01_RB(1024).png …
[8/12] 0.9_0.6_60_01_RB(1024).png …
[9/12] 0.9_0.6_90_01_RB(1024).png …
[10/12] 0.9_0.9_30_01_RB(1024).png …
[11/12] 0.9_0.9_60_01_RB(1024).png …
[12/12] 0.9_0.9_90_01_RB(1024).png …
✅ 完成！共處理 12 張圖。
📄 CSV：C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0.6_0.9\0810ResultFigures\batch_edge_summary.csv
🖼 圖片輸出資料夾：C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0.6_0.9\0810ResultFigures
🧪 Debug 二值化輸出：C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\DOE_RB\0.6_0.9\0810ResultFigures\_debug_binary
